# 01 — Comprensión de datos

## Proyecto: HR Process Automation Scanner


El objetivo de este notebook es explorar los datasets descargados para decidir cuál será la fuente principal del proyecto.

Este proyecto busca construir un recomendador de automatización de procesos HR basado en datos, Machine Learning y lógica de negocio.

La solución final permitirá:

- analizar procesos de HR Operations;
- calcular un score de oportunidad de automatización;
- clasificar procesos según prioridad;
- recomendar el tipo de solución más adecuada;
- estimar impacto operativo;
- construir una aplicación en Streamlit orientada a la toma de decisiones.

## Diferencia importante:

No todos los procesos deben automatizarse con IA. Este proyecto recomendará dónde tiene sentido aplicar IA y dónde no.

Algunos procesos sí son buenos candidatos:

repetitivos;
frecuentes;
basados en reglas;
con bajo riesgo;
con datos disponibles;
con mucha carga manual.

Otros no conviene automatizarlos totalmente:

casos legales complejos;
employee relations sensibles;
conflictos laborales;
decisiones disciplinarias;
situaciones con alta carga humana o ética.

## El producto final

El resultado será una app en Streamlit tipo:

HR Process Automation Scanner

Donde el usuario podrá:

Ver ranking de procesos HR.
Identificar quick wins de automatización.
Simular un nuevo proceso.
Recibir una recomendación automática.
Ver qué tipo de tecnología aplicar.
Estimar impacto operativo.


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_PATH / "raw"
PROCESSED_DATA_PATH = DATA_PATH / "processed"
FINAL_DATA_PATH = DATA_PATH / "final"

print("Carpeta actual:", Path.cwd())
print("Raíz del proyecto:", PROJECT_ROOT)
print("Ruta raw:", RAW_DATA_PATH)
print("Existe data/raw:", RAW_DATA_PATH.exists())

Carpeta actual: /Users/gabrielbohorquez/Desktop/Ironhack/PROYECTO_FINAL/HR-Process-Automation-Scanner/notebooks
Raíz del proyecto: /Users/gabrielbohorquez/Desktop/Ironhack/PROYECTO_FINAL/HR-Process-Automation-Scanner
Ruta raw: /Users/gabrielbohorquez/Desktop/Ironhack/PROYECTO_FINAL/HR-Process-Automation-Scanner/data/raw
Existe data/raw: True


In [2]:
for path in RAW_DATA_PATH.rglob("*"):
    if path.is_file():
        print(path)

/Users/gabrielbohorquez/Desktop/Ironhack/PROYECTO_FINAL/HR-Process-Automation-Scanner/data/raw/customer_support_tickets_200k.csv
/Users/gabrielbohorquez/Desktop/Ironhack/PROYECTO_FINAL/HR-Process-Automation-Scanner/data/raw/customer_support_tickets.csv
/Users/gabrielbohorquez/Desktop/Ironhack/PROYECTO_FINAL/HR-Process-Automation-Scanner/data/raw/.DS_Store
/Users/gabrielbohorquez/Desktop/Ironhack/PROYECTO_FINAL/HR-Process-Automation-Scanner/data/raw/helpdesk_event_log.csv


In [3]:
# Detectar y cargar los CSV
csv_files = list(RAW_DATA_PATH.rglob("*.csv"))

csv_files

[PosixPath('/Users/gabrielbohorquez/Desktop/Ironhack/PROYECTO_FINAL/HR-Process-Automation-Scanner/data/raw/customer_support_tickets_200k.csv'),
 PosixPath('/Users/gabrielbohorquez/Desktop/Ironhack/PROYECTO_FINAL/HR-Process-Automation-Scanner/data/raw/customer_support_tickets.csv'),
 PosixPath('/Users/gabrielbohorquez/Desktop/Ironhack/PROYECTO_FINAL/HR-Process-Automation-Scanner/data/raw/helpdesk_event_log.csv')]

In [4]:
# Crear función de inspección
def inspect_dataset(file_path, nrows=5):
    """
    Carga e inspecciona un dataset CSV.

    Parámetros
    ----------
    file_path : str or Path
        Ruta del archivo CSV.
    nrows : int
        Número de filas a mostrar como vista previa.

    Devuelve
    --------
    pd.DataFrame
        DataFrame cargado.
    """
    file_path = Path(file_path)

    print("=" * 100)
    print(f"ARCHIVO: {file_path.name}")
    print("=" * 100)

    try:
        df = pd.read_csv(file_path)

        print(f"Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas")

        print("\nColumnas:")
        print(df.columns.tolist())

        print("\nTipos de datos:")
        display(df.dtypes.to_frame("tipo_dato"))

        print("\nValores nulos:")
        display(df.isna().sum().to_frame("valores_nulos"))

        print("\nVista previa:")
        display(df.head(nrows))

        return df

    except Exception as error:
        print(f"Error al cargar el archivo: {error}")
        return None

In [5]:
# Cargar todos los datasets
datasets = {}

for file_path in csv_files:
    dataset_name = file_path.stem.lower().replace(" ", "_").replace("-", "_")
    datasets[dataset_name] = inspect_dataset(file_path)

ARCHIVO: customer_support_tickets_200k.csv
Dimensiones: 200000 filas y 30 columnas

Columnas:
['ticket_id', 'customer_name', 'customer_email', 'product', 'category', 'issue_description', 'resolution_notes', 'priority', 'status', 'channel', 'region', 'customer_age', 'customer_gender', 'subscription_type', 'customer_tenure_months', 'previous_tickets', 'customer_satisfaction_score', 'first_response_time_hours', 'resolution_time_hours', 'ticket_created_date', 'ticket_resolved_date', 'escalated', 'sla_breached', 'operating_system', 'browser', 'payment_method', 'language', 'preferred_contact_time', 'issue_complexity_score', 'customer_segment']

Tipos de datos:


,tipo_dato
ticket_id,int64
customer_name,object
customer_email,object
product,object
category,object
issue_description,object
resolution_notes,object
priority,object
status,object
channel,object



Valores nulos:


,valores_nulos
ticket_id,0
customer_name,0
customer_email,0
product,0
category,0
issue_description,0
resolution_notes,0
priority,0
status,0
channel,0



Vista previa:


...

,tipo_dato
CaseID,int64
ActivityID,int64
CompleteTimestamp,object



Valores nulos:


,valores_nulos
CaseID,0
ActivityID,0
CompleteTimestamp,0



Vista previa:


,CaseID,ActivityID,CompleteTimestamp
0,2,1,2012-04-03 16:55:38
1,2,8,2012-04-03 16:55:53
2,2,6,2012-04-05 17:15:52
3,3,1,2010-10-29 18:14:06
4,3,8,2010-11-04 01:16:11


In [6]:
# Resumen comparativo:
dataset_summary = []

for name, df in datasets.items():
    if df is not None:
        dataset_summary.append({
            "nombre_dataset": name,
            "filas": df.shape[0],
            "columnas": df.shape[1],
            "nombres_columnas": ", ".join(df.columns.astype(str))
        })

dataset_summary_df = pd.DataFrame(dataset_summary)

dataset_summary_df

,nombre_dataset,filas,columnas,nombres_columnas
0,customer_support_tickets_200k,200000,30,"ticket_id, customer_name, customer_email, product, category, issue_description, resolution_notes, priority, status, ..."
1,customer_support_tickets,8469,17,"Ticket ID, Customer Name, Customer Email, Customer Age, Customer Gender, Product Purchased, Date of Purchase, Ticket..."
2,helpdesk_event_log,13710,3,"CaseID, ActivityID, CompleteTimestamp"


In [7]:
# Ver columnas por dataset
for name, df in datasets.items():
    print("\n" + "=" * 100)
    print(f"DATASET: {name}")
    print("=" * 100)
    print(df.columns.tolist())


DATASET: customer_support_tickets_200k
['ticket_id', 'customer_name', 'customer_email', 'product', 'category', 'issue_description', 'resolution_notes', 'priority', 'status', 'channel', 'region', 'customer_age', 'customer_gender', 'subscription_type', 'customer_tenure_months', 'previous_tickets', 'customer_satisfaction_score', 'first_response_time_hours', 'resolution_time_hours', 'ticket_created_date', 'ticket_resolved_date', 'escalated', 'sla_breached', 'operating_system', 'browser', 'payment_method', 'language', 'preferred_contact_time', 'issue_complexity_score', 'customer_segment']

DATASET: customer_support_tickets
['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']

DATASET: helpdesk_event_log
['CaseID', 'Activit

## Decisión inicial

Usaremos como dataset principal: **customer_support_tickets_200k**

| Criterio | Evaluación |
|----------|------------|
| Volumen | Muy alto: 200.000 filas |
| Columnas | 30 columnas |
| Valor operativo | Alto |
| Potencial para ML | Alto |
| Potencial para simulación | Alto |
| Potencial para app Streamlit | Alto |

---

## Rol de cada dataset

| Dataset | Filas | Columnas | Uso recomendado |
|---------|-------|----------|-----------------|
| customer_support_tickets_200k | 200.000 | 30 | Dataset principal |
| customer_support_tickets | 8.469 | 17 | Dataset secundario / comparación / backup |
| helpdesk_event_log | 13.710 | 3 | Dataset complementario para lógica de proceso |

---

## Cómo interpretaremos cada dataset

### 1. customer_support_tickets_200k

Este será nuestro equivalente a tickets internos de HR Operations.

| Dataset original | Interpretación en el proyecto |
|------------------|-------------------------------|
| ticket | Ticket interno de HR |
| category | Proceso HR |
| issue_description | Descripción del caso |
| priority | Prioridad del ticket |
| status | Estado del caso |
| channel | Canal de entrada |
| region | Región o mercado |
| resolution_notes | Notas de resolución |
| customer_age / gender | Variables que probablemente eliminaremos |
| subscription_type | Podría transformarse en tipo de empleado o segmento |
| customer_tenure_months | Podría transformarse en antigüedad del empleado |

### 2. customer_support_tickets

Lo usaremos solo si necesitamos comparar estructura o enriquecer ideas. Tiene menos filas y menos columnas, así que no será el principal.

### 3. helpdesk_event_log

Tiene solo tres columnas: `CaseID`, `ActivityID`, `CompleteTimestamp`.

No sirve como dataset principal, pero es útil para calcular lógica de proceso:

| Uso |
|-----|
| Número de actividades por caso |
| Duración total del caso |
| Cantidad de pasos |
| Complejidad del flujo |
| Posible estandarización |
| Variabilidad entre casos |

## Inspeccionar dataset principal:

Exploraremos:

- todas sus columnas completas;

- tipos de datos;

- nulos;

- valores únicos;

- primeras filas;

- si tiene fechas;


si tiene variables útiles para crear:

- volumen mensual;

- tiempo medio;

- tasa de errores;

- tasa de resolución;

- prioridad;

- canal;

- categoría;

- posible score de automatización.

In [8]:
# Extraer el dataset principal:
df_main = datasets["customer_support_tickets_200k"].copy()

df_main.shape

(200000, 30)

In [9]:
# Ver todas las columnas completas:
df_main.columns.tolist()

['ticket_id',
 'customer_name',
 'customer_email',
 'product',
 'category',
 'issue_description',
 'resolution_notes',
 'priority',
 'status',
 'channel',
 'region',
 'customer_age',
 'customer_gender',
 'subscription_type',
 'customer_tenure_months',
 'previous_tickets',
 'customer_satisfaction_score',
 'first_response_time_hours',
 'resolution_time_hours',
 'ticket_created_date',
 'ticket_resolved_date',
 'escalated',
 'sla_breached',
 'operating_system',
 'browser',
 'payment_method',
 'language',
 'preferred_contact_time',
 'issue_complexity_score',
 'customer_segment']

In [10]:
# Primeras filas:
df_main.head()

,ticket_id,customer_name,customer_email,product,category,issue_description,resolution_notes,priority,status,channel,region,customer_age,customer_gender,subscription_type,customer_tenure_months,previous_tickets,customer_satisfaction_score,first_response_time_hours,resolution_time_hours,ticket_created_date,ticket_resolved_date,escalated,sla_breached,operating_system,browser,payment_method,language,preferred_contact_time,issue_complexity_score,customer_segment
0,1,Patricia Smith,patricia.smith760@outlook.com,Web Portal,Account Suspension,The payment was deducted from my bank account but the transaction shows failed.,Data synchronization restored after backend service restart.,Urgent,Open,Email,North America,31,Male,Free,36,6,5,30.50,108.36,2023-05-17,2023-05-20,No,Yes,MacOS,Edge,PayPal,French,Afternoon,4,Small Business
1,2,Patricia Williams,patricia.williams390@gmail.com,Mobile App,Performance Issue,I found a bug in the latest update affecting report generation.,Provided step-by-step troubleshooting instructions and issue resolved.,Urgent,Closed,Email,South America,23,Other,Premium,54,20,5,63.81,87.43,2024-01-06,2024-01-19,Yes,Yes,Windows,Firefox,PayPal,English,Afternoon,2,Small Business
2,3,William Anderson,william.anderson651@outlook.com,Web Portal,Performance Issue,The application crashes whenever I try to upload a file.,Provided step-by-step troubleshooting instructions and issue resolved.,Medium,Closed,Chat,Europe,47,Female,Premium,60,20,5,16.20,78.50,2022-11-30,2022-12-05,Yes,Yes,Windows,Safari,Bank Transfer,French,Morning,4,Corporate
3,4,David Miller,david.miller672@icloud.com,Payment Gateway,Subscription Cancellation,My subscription was cancelled without my request and I need clarification.,Provided step-by-step troubleshooting instructions and issue resolved.,Medium,Closed,Social Media,Asia,65,Other,Enterprise,58,18,4,26.38,239.36,2024-03-21,2024-04-04,Yes,No,Windows,Chrome,Credit Card,Spanish,Afternoon,7,Corporate
4,5,Robert Gonzalez,robert.gonzalez391@hotmail.com,Web Portal,Feature Request,The system is not syncing data across devices properly.,We have reset the account credentials and advised the customer to retry login.,High,Pending Customer,Email,Asia,45,Male,Enterprise,1,8,5,54.98,122.34,2024-08-16,2024-08-24,Yes,No,Linux,NaN,Debit Card,Spanish,Evening,3,Corporate


In [11]:
# Ver tipos de datos:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 30 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   ticket_id                    200000 non-null  int64  
 1   customer_name                200000 non-null  object 
 2   customer_email               200000 non-null  object 
 3   product                      200000 non-null  object 
 4   category                     200000 non-null  object 
 5   issue_description            200000 non-null  object 
 6   resolution_notes             200000 non-null  object 
 7   priority                     200000 non-null  object 
 8   status                       200000 non-null  object 
 9   channel                      200000 non-null  object 
 10  region                       200000 non-null  object 
 11  customer_age                 200000 non-null  int64  
 12  customer_gender              200000 non-null  object 
 13 

In [12]:
df_main.dtypes.to_frame("tipo_dato")

,tipo_dato
ticket_id,int64
customer_name,object
customer_email,object
product,object
category,object
issue_description,object
resolution_notes,object
priority,object
status,object
channel,object


In [13]:
# Ver valores nulos:
missing_summary = (
    df_main
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "columna", 0: "valores_nulos"})
)

missing_summary["porcentaje_nulos"] = (
    missing_summary["valores_nulos"] / len(df_main) * 100
).round(2)

missing_summary.sort_values("porcentaje_nulos", ascending=False)

,columna,valores_nulos,porcentaje_nulos
24,browser,40023,20.01
0,ticket_id,0,0.00
1,customer_name,0,0.00
28,issue_complexity_score,0,0.00
27,preferred_contact_time,0,0.00
26,language,0,0.00
25,payment_method,0,0.00
23,operating_system,0,0.00
22,sla_breached,0,0.00
21,escalated,0,0.00


In [14]:
# Ver variables categóricas:
categorical_columns = df_main.select_dtypes(include="object").columns.tolist()

categorical_columns

['customer_name',
 'customer_email',
 'product',
 'category',
 'issue_description',
 'resolution_notes',
 'priority',
 'status',
 'channel',
 'region',
 'customer_gender',
 'subscription_type',
 'ticket_created_date',
 'ticket_resolved_date',
 'escalated',
 'sla_breached',
 'operating_system',
 'browser',
 'payment_method',
 'language',
 'preferred_contact_time',
 'customer_segment']

In [15]:
for column in categorical_columns:
    print("\n" + "=" * 80)
    print(f"COLUMNA: {column}")
    print("=" * 80)
    print(df_main[column].value_counts(dropna=False).head(20))


COLUMNA: customer_name
customer_name
Jessica Rodriguez    568
Linda Williams       566
Linda Jackson        560
Robert Anderson      556
David Johnson        554
Barbara Gonzalez     553
James Moore          546
Elizabeth Smith      545
Sarah Jackson        545
Richard Wilson       545
Michael Anderson     545
John Brown           543
Linda Martin         542
Joseph Hernandez     541
Thomas Wilson        540
Sarah Rodriguez      539
Thomas Thomas        539
Elizabeth Garcia     538
Elizabeth Johnson    537
Karen Jones          536
Name: count, dtype: int64

COLUMNA: customer_email
customer_email
john.gonzalez711@gmail.com        4
barbara.martin919@yahoo.com       4
william.moore661@company.com      4
mary.moore262@yahoo.com           4
richard.anderson171@icloud.com    4
susan.wilson217@gmail.com         4
jennifer.wilson148@outlook.com    3
joseph.thomas467@gmail.com        3
joseph.jackson739@hotmail.com     3
patricia.smith434@hotmail.com     3
sarah.martinez240@company.com     3


Vemos valores más frecuentes de columnas como:

- category;

- priority;

- status;

- channel;

- region;

- etc.

In [16]:
# Ver variables numéricas:
numeric_columns = df_main.select_dtypes(include=["int64", "float64"]).columns.tolist()

numeric_columns

['ticket_id',
 'customer_age',
 'customer_tenure_months',
 'previous_tickets',
 'customer_satisfaction_score',
 'first_response_time_hours',
 'resolution_time_hours',
 'issue_complexity_score']

In [17]:
df_main[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
ticket_id,200000.0,100000.500000,57735.171256,1.0,50000.75,100000.50,150000.25,200000.0
customer_age,200000.0,46.469040,16.754304,18.0,32.00,46.00,61.00,75.0
customer_tenure_months,200000.0,30.377675,17.326224,1.0,15.00,30.00,45.00,60.0
previous_tickets,200000.0,9.992115,6.053142,0.0,5.00,10.00,15.00,20.0
customer_satisfaction_score,200000.0,3.001065,1.412273,1.0,2.00,3.00,4.00,5.0
first_response_time_hours,200000.0,36.309180,20.654575,0.5,18.46,36.31,54.22,72.0
resolution_time_hours,200000.0,120.539718,68.969143,1.0,60.81,120.45,180.26,240.0
issue_complexity_score,200000.0,5.501365,2.875587,1.0,3.00,6.00,8.00,10.0


In [18]:
# Guardamos un resumen de columnas - Esto permitirá documentar el notebook:
column_summary = pd.DataFrame({
    "columna": df_main.columns,
    "tipo_dato": df_main.dtypes.astype(str).values,
    "valores_nulos": df_main.isna().sum().values,
    "porcentaje_nulos": (df_main.isna().sum().values / len(df_main) * 100).round(2),
    "valores_unicos": df_main.nunique().values
})

column_summary

,columna,tipo_dato,valores_nulos,porcentaje_nulos,valores_unicos
0,ticket_id,int64,0,0.00,200000
1,customer_name,object,0,0.00,400
2,customer_email,object,0,0.00,191751
3,product,object,0,0.00,10
4,category,object,0,0.00,10
5,issue_description,object,0,0.00,10
6,resolution_notes,object,0,0.00,10
7,priority,object,0,0.00,4
8,status,object,0,0.00,5
9,channel,object,0,0.00,5


## Decisión técnica

Este dataset sirve para el proyecto.

Pero ahora debemos transformarlo.

El dataset original es de soporte al cliente. Por eso, lo vamos a convertir en una versión orientada a HR Operations.

Para eso crearemos un nuevo DataFrame llamado:

df_hr

Este será el dataset de trabajo del proyecto.

## Adaptación inicial del dataset al contexto de HR Operations

El dataset original contiene tickets de soporte al cliente. Para este proyecto, se reutilizará como base operativa para simular tickets internos de HR Operations.

La lógica de adaptación será la siguiente:

- cada ticket representará una solicitud interna de HR;
- la categoría original se transformará en un proceso de HR;
- el producto original se interpretará como sistema o plataforma relacionada;
- las variables de prioridad, canal, región, escalación, SLA y tiempo de resolución se conservarán porque son relevantes para operaciones;
- las variables personales del cliente que no aportan valor directo al objetivo del proyecto serán descartadas o transformadas.

El objetivo es construir una base analítica que permita evaluar qué procesos de HR tienen mayor oportunidad de automatización u optimización con IA.

In [20]:
# Crear una copia del dataset principal para adaptarlo al contexto de HR Operations
df_hr = df_main.copy()

# Renombrar columnas para alinearlas con el caso de negocio
df_hr = df_hr.rename(columns={
    "ticket_id": "case_id",
    "product": "hr_system",
    "category": "hr_process",
    "issue_description": "case_description",
    "priority": "case_priority",
    "status": "case_status",
    "channel": "contact_channel",
    "customer_tenure_months": "employee_tenure_months",
    "previous_tickets": "previous_cases",
    "customer_satisfaction_score": "employee_satisfaction_score",
    "issue_complexity_score": "process_complexity_score",
    "customer_segment": "employee_segment"
})

df_hr.head()

,case_id,customer_name,customer_email,hr_system,hr_process,case_description,resolution_notes,case_priority,case_status,contact_channel,region,customer_age,customer_gender,subscription_type,employee_tenure_months,previous_cases,employee_satisfaction_score,first_response_time_hours,resolution_time_hours,ticket_created_date,ticket_resolved_date,escalated,sla_breached,operating_system,browser,payment_method,language,preferred_contact_time,process_complexity_score,employee_segment
0,1,Patricia Smith,patricia.smith760@outlook.com,Web Portal,Account Suspension,The payment was deducted from my bank account but the transaction shows failed.,Data synchronization restored after backend service restart.,Urgent,Open,Email,North America,31,Male,Free,36,6,5,30.50,108.36,2023-05-17,2023-05-20,No,Yes,MacOS,Edge,PayPal,French,Afternoon,4,Small Business
1,2,Patricia Williams,patricia.williams390@gmail.com,Mobile App,Performance Issue,I found a bug in the latest update affecting report generation.,Provided step-by-step troubleshooting instructions and issue resolved.,Urgent,Closed,Email,South America,23,Other,Premium,54,20,5,63.81,87.43,2024-01-06,2024-01-19,Yes,Yes,Windows,Firefox,PayPal,English,Afternoon,2,Small Business
2,3,William Anderson,william.anderson651@outlook.com,Web Portal,Performance Issue,The application crashes whenever I try to upload a file.,Provided step-by-step troubleshooting instructions and issue resolved.,Medium,Closed,Chat,Europe,47,Female,Premium,60,20,5,16.20,78.50,2022-11-30,2022-12-05,Yes,Yes,Windows,Safari,Bank Transfer,French,Morning,4,Corporate
3,4,David Miller,david.miller672@icloud.com,Payment Gateway,Subscription Cancellation,My subscription was cancelled without my request and I need clarification.,Provided step-by-step troubleshooting instructions and issue resolved.,Medium,Closed,Social Media,Asia,65,Other,Enterprise,58,18,4,26.38,239.36,2024-03-21,2024-04-04,Yes,No,Windows,Chrome,Credit Card,Spanish,Afternoon,7,Corporate
4,5,Robert Gonzalez,robert.gonzalez391@hotmail.com,Web Portal,Feature Request,The system is not syncing data across devices properly.,We have reset the account credentials and advised the customer to retry login.,High,Pending Customer,Email,Asia,45,Male,Enterprise,1,8,5,54.98,122.34,2024-08-16,2024-08-24,Yes,No,Linux,NaN,Debit Card,Spanish,Evening,3,Corporate


In [21]:
# Revisar columnas nuevas:
for index, column in enumerate(df_hr.columns, start=1):
    print(f"{index}. {column}")

1. case_id
2. customer_name
3. customer_email
4. hr_system
5. hr_process
6. case_description
7. resolution_notes
8. case_priority
9. case_status
10. contact_channel
11. region
12. customer_age
13. customer_gender
14. subscription_type
15. employee_tenure_months
16. previous_cases
17. employee_satisfaction_score
18. first_response_time_hours
19. resolution_time_hours
20. ticket_created_date
21. ticket_resolved_date
22. escalated
23. sla_breached
24. operating_system
25. browser
26. payment_method
27. language
28. preferred_contact_time
29. process_complexity_score
30. employee_segment


In [22]:
# Eliminar columnas que no aportan al proyecto:
columns_to_drop = [
    "customer_name",
    "customer_email",
    "customer_age",
    "customer_gender",
    "operating_system",
    "browser",
    "payment_method"
]

df_hr = df_hr.drop(columns=columns_to_drop)

df_hr.shape

(200000, 23)

In [23]:
# Revisar resultado final:
df_hr.head()

,case_id,hr_system,hr_process,case_description,resolution_notes,case_priority,case_status,contact_channel,region,subscription_type,employee_tenure_months,previous_cases,employee_satisfaction_score,first_response_time_hours,resolution_time_hours,ticket_created_date,ticket_resolved_date,escalated,sla_breached,language,preferred_contact_time,process_complexity_score,employee_segment
0,1,Web Portal,Account Suspension,The payment was deducted from my bank account but the transaction shows failed.,Data synchronization restored after backend service restart.,Urgent,Open,Email,North America,Free,36,6,5,30.50,108.36,2023-05-17,2023-05-20,No,Yes,French,Afternoon,4,Small Business
1,2,Mobile App,Performance Issue,I found a bug in the latest update affecting report generation.,Provided step-by-step troubleshooting instructions and issue resolved.,Urgent,Closed,Email,South America,Premium,54,20,5,63.81,87.43,2024-01-06,2024-01-19,Yes,Yes,English,Afternoon,2,Small Business
2,3,Web Portal,Performance Issue,The application crashes whenever I try to upload a file.,Provided step-by-step troubleshooting instructions and issue resolved.,Medium,Closed,Chat,Europe,Premium,60,20,5,16.20,78.50,2022-11-30,2022-12-05,Yes,Yes,French,Morning,4,Corporate
3,4,Payment Gateway,Subscription Cancellation,My subscription was cancelled without my request and I need clarification.,Provided step-by-step troubleshooting instructions and issue resolved.,Medium,Closed,Social Media,Asia,Enterprise,58,18,4,26.38,239.36,2024-03-21,2024-04-04,Yes,No,Spanish,Afternoon,7,Corporate
4,5,Web Portal,Feature Request,The system is not syncing data across devices properly.,We have reset the account credentials and advised the customer to retry login.,High,Pending Customer,Email,Asia,Enterprise,1,8,5,54.98,122.34,2024-08-16,2024-08-24,Yes,No,Spanish,Evening,3,Corporate


In [24]:
df_hr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 23 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   case_id                      200000 non-null  int64  
 1   hr_system                    200000 non-null  object 
 2   hr_process                   200000 non-null  object 
 3   case_description             200000 non-null  object 
 4   resolution_notes             200000 non-null  object 
 5   case_priority                200000 non-null  object 
 6   case_status                  200000 non-null  object 
 7   contact_channel              200000 non-null  object 
 8   region                       200000 non-null  object 
 9   subscription_type            200000 non-null  object 
 10  employee_tenure_months       200000 non-null  int64  
 11  previous_cases               200000 non-null  int64  
 12  employee_satisfaction_score  200000 non-null  int64  
 13 

# ¿Por qué hacemos esto?

Porque el proyecto no va de clientes, emails, navegadores o métodos de pago.

El proyecto va de:

* procesos HR

* volumen operativo

* tiempos

* SLA

* complejidad

* escalaciones

* prioridad

* automatización


Así que limpiamos la estructura para que el dataset sea un dataset profesional de HR Operations.

## Transformación de procesos:
Hemos conseguido "hr_process" . Pero sus valores todavía vienen del dataset original, por ejemplo:

- Account Suspension
- Performance Issue
- Subscription Cancellation
- Feature Request

Eso todavía no parece HR.

Tenemos que transformarlo en procesos como:

* HRIS Access Issue

* Payroll Support

* Employee Data Update

* Benefits Request

* Onboarding Documentation

* Leave of Absence

* Contract Management

* Internal Transfer

* Training Request

* Compliance Documentation


In [25]:
# Ver valores únicos actuales de hr_process:
df_hr["hr_process"].value_counts()

hr_process
Feature Request              20169
Subscription Cancellation    20096
Performance Issue            20074
Security Concern             20040
Login Issue                  20002
Payment Problem              19997
Bug Report                   19981
Refund Request               19900
Data Sync Issue              19877
Account Suspension           19864
Name: count, dtype: int64

Aquí podemos visualizar qué categorías originales existen y cuántos casos tiene cada una.

In [26]:
# Ver valores únicos de hr_system:
df_hr["hr_system"].value_counts()

hr_system
Billing System          20111
CRM Platform            20097
E-commerce Store        20095
Cloud Storage           20056
Mobile App              20043
Analytics Dashboard     19974
Web Portal              19954
Payment Gateway         19952
Subscription Service    19883
API Service             19835
Name: count, dtype: int64

Esta información representa el sistema o plataforma relacionada con el caso.

In [27]:
# Ver valores únicos de variables operativas clave:
columns_to_review = [
    "hr_process",
    "hr_system",
    "case_priority",
    "case_status",
    "contact_channel",
    "region",
    "subscription_type",
    "escalated",
    "sla_breached",
    "language",
    "preferred_contact_time",
    "employee_segment"
]

for column in columns_to_review:
    print("\n" + "=" * 80)
    print(f"COLUMNA: {column}")
    print("=" * 80)
    print(df_hr[column].value_counts(dropna=False))


COLUMNA: hr_process
hr_process
Feature Request              20169
Subscription Cancellation    20096
Performance Issue            20074
Security Concern             20040
Login Issue                  20002
Payment Problem              19997
Bug Report                   19981
Refund Request               19900
Data Sync Issue              19877
Account Suspension           19864
Name: count, dtype: int64

COLUMNA: hr_system
hr_system
Billing System          20111
CRM Platform            20097
E-commerce Store        20095
Cloud Storage           20056
Mobile App              20043
Analytics Dashboard     19974
Web Portal              19954
Payment Gateway         19952
Subscription Service    19883
API Service             19835
Name: count, dtype: int64

COLUMNA: case_priority
case_priority
High      50241
Urgent    50143
Medium    49854
Low       49762
Name: count, dtype: int64

COLUMNA: case_status
case_status
In Progress         40065
Closed              40029
Pending Customer    40

## Hasta aquí vemos que:

hr_process tiene 10 categorías originales:

1. Feature Request
2. Subscription Cancellation
3. Performance Issue
4. Security Concern
5. Login Issue
6. Payment Problem
7. Bug Report
8. Refund Request
9. Data Sync Issue
10. Account Suspension

hr_system tiene 10 sistemas originales:

1. Billing System
2. CRM Platform
3. E-commerce Store
4. Cloud Storage
5. Mobile App
6. Analytics Dashboard
7. Web Portal
8. Payment Gateway
9. Subscription Service
10. API Service

Ahora vamos a hacer la adaptación profesional a HR.

## 1.5 Adaptación de categorías originales a procesos de HR Operations

El dataset original pertenece a un contexto de soporte al cliente. Para alinear el proyecto con el objetivo de People Analytics y HR Operations, se transformarán las categorías originales en procesos equivalentes de Recursos Humanos.

Esta adaptación no modifica la estructura operativa del dataset: se conservan variables como prioridad, estado, canal, región, escalación, SLA, tiempo de respuesta y tiempo de resolución.

El objetivo es construir una capa de negocio que permita interpretar cada ticket como una solicitud interna de HR Operations.

In [29]:
# Mapear procesos originales a procesos HR:
# Crear mapeo de categorías originales a procesos reales de HR Operations
hr_process_mapping = {
    "Feature Request": "HR System Enhancement Request",
    "Subscription Cancellation": "Benefits Cancellation Request",
    "Performance Issue": "HR System Performance Issue",
    "Security Concern": "Compliance & Access Security Review",
    "Login Issue": "HRIS Login Issue",
    "Payment Problem": "Payroll Support Request",
    "Bug Report": "HR Platform Bug Report",
    "Refund Request": "Benefits Reimbursement Request",
    "Data Sync Issue": "Employee Data Update",
    "Account Suspension": "HRIS Access Suspension"
}

# Crear nueva columna con el proceso HR adaptado
df_hr["hr_process_name"] = df_hr["hr_process"].map(hr_process_mapping)

# Revisar resultado
df_hr[["hr_process", "hr_process_name"]].drop_duplicates().sort_values("hr_process")


,hr_process,hr_process_name
0,Account Suspension,HRIS Access Suspension
22,Bug Report,HR Platform Bug Report
28,Data Sync Issue,Employee Data Update
4,Feature Request,HR System Enhancement Request
19,Login Issue,HRIS Login Issue
5,Payment Problem,Payroll Support Request
1,Performance Issue,HR System Performance Issue
15,Refund Request,Benefits Reimbursement Request
6,Security Concern,Compliance & Access Security Review
3,Subscription Cancellation,Benefits Cancellation Request


In [30]:
# Crear mapeo de sistemas originales a sistemas/plataformas de HR
hr_system_mapping = {
    "Billing System": "Payroll System",
    "CRM Platform": "HR Case Management Platform",
    "E-commerce Store": "Benefits Portal",
    "Cloud Storage": "HR Document Management System",
    "Mobile App": "Employee Mobile App",
    "Analytics Dashboard": "People Analytics Dashboard",
    "Web Portal": "Employee Self-Service Portal",
    "Payment Gateway": "Payroll Payment Gateway",
    "Subscription Service": "Benefits Administration System",
    "API Service": "HRIS Integration API"
}

# Crear nueva columna con el sistema HR adaptado
df_hr["hr_system_name"] = df_hr["hr_system"].map(hr_system_mapping)

# Revisar resultado
df_hr[["hr_system", "hr_system_name"]].drop_duplicates().sort_values("hr_system")

,hr_system,hr_system_name
8,API Service,HRIS Integration API
18,Analytics Dashboard,People Analytics Dashboard
9,Billing System,Payroll System
30,CRM Platform,HR Case Management Platform
12,Cloud Storage,HR Document Management System
14,E-commerce Store,Benefits Portal
1,Mobile App,Employee Mobile App
3,Payment Gateway,Payroll Payment Gateway
5,Subscription Service,Benefits Administration System
0,Web Portal,Employee Self-Service Portal


In [31]:
# Verificar si el mapeo dejó valores sin traducir:
# Verificar valores no mapeados
unmapped_processes = df_hr[df_hr["hr_process_name"].isna()]["hr_process"].unique()
unmapped_systems = df_hr[df_hr["hr_system_name"].isna()]["hr_system"].unique()

print("Procesos sin mapear:")
print(unmapped_processes)

print("\nSistemas sin mapear:")
print(unmapped_systems)

Procesos sin mapear:
[]

Sistemas sin mapear:
[]


In [32]:
# Revisar distribución de procesos HR adaptados:
df_hr["hr_process_name"].value_counts()

hr_process_name
HR System Enhancement Request          20169
Benefits Cancellation Request          20096
HR System Performance Issue            20074
Compliance & Access Security Review    20040
HRIS Login Issue                       20002
Payroll Support Request                19997
HR Platform Bug Report                 19981
Benefits Reimbursement Request         19900
Employee Data Update                   19877
HRIS Access Suspension                 19864
Name: count, dtype: int64

In [33]:
# Revisar distribución de sistemas HR adaptados:
df_hr["hr_system_name"].value_counts()

hr_system_name
Payroll System                    20111
HR Case Management Platform       20097
Benefits Portal                   20095
HR Document Management System     20056
Employee Mobile App               20043
People Analytics Dashboard        19974
Employee Self-Service Portal      19954
Payroll Payment Gateway           19952
Benefits Administration System    19883
HRIS Integration API              19835
Name: count, dtype: int64

In [34]:
# Tabla resumen de procesos HR:
hr_process_summary = (
    df_hr
    .groupby("hr_process_name")
    .agg(
        total_cases=("case_id", "count"),
        avg_resolution_time_hours=("resolution_time_hours", "mean"),
        avg_first_response_time_hours=("first_response_time_hours", "mean"),
        avg_complexity_score=("process_complexity_score", "mean"),
        avg_satisfaction_score=("employee_satisfaction_score", "mean")
    )
    .reset_index()
)

hr_process_summary = hr_process_summary.sort_values("total_cases", ascending=False)

hr_process_summary

,hr_process_name,total_cases,avg_resolution_time_hours,avg_first_response_time_hours,avg_complexity_score,avg_satisfaction_score
5,HR System Enhancement Request,20169,120.610610,36.197824,5.513362,2.996034
0,Benefits Cancellation Request,20096,120.543078,36.469808,5.511943,3.012042
6,HR System Performance Issue,20074,120.728006,36.435281,5.514995,2.995467
2,Compliance & Access Security Review,20040,120.362533,36.256427,5.467814,2.993513
8,HRIS Login Issue,20002,120.661216,36.308648,5.506649,2.999000
9,Payroll Support Request,19997,120.939464,36.171778,5.512427,3.007701
4,HR Platform Bug Report,19981,120.318886,36.133836,5.479155,3.000500
1,Benefits Reimbursement Request,19900,119.794275,36.679083,5.483769,3.010704
3,Employee Data Update,19877,121.021393,36.195242,5.512200,3.008502
7,HRIS Access Suspension,19864,120.414983,36.244197,5.511226,2.987213


## Proceso hasta aquí:

Ahora el dataset empieza a tener sentido para el proyecto:

| Antes | Ahora |
|-------|-------|
| category | hr_process_name |
| product | hr_system_name |
| customer support | HR Operations case management |
| customer satisfaction | employee satisfaction |
| ticket | HR case |

## Problema técnico importante

Ahora mismo, si agrupamos solo por hr_process_name, tenemos únicamente 10 procesos.

No sirve bien para Machine Learning, es decir, para construir un recomendador fuerte, ya que 10 filas son muy pocas para entrenar un modelo.

Por eso haremos algo más profesional:

Crearemos unidades operativas de proceso


## Creación de unidades operativas de proceso

Para que el análisis sea útil desde una perspectiva operativa, no se analizarán únicamente los procesos generales de HR.

En su lugar, se crearán unidades operativas de proceso combinando:

- proceso HR;
- sistema HR;
- región;
- canal de contacto;
- prioridad del caso.

Esto permite evaluar oportunidades de automatización de forma más granular.

Por ejemplo, no es lo mismo analizar "Payroll Support Request" de forma general que analizar "Payroll Support Request en Europa, recibido por email, con prioridad alta".

Analizaremos combinaciones como:

Payroll Support Request + Payroll System + Europe + Email
Payroll Support Request + Payroll System + Asia + Chat
HRIS Login Issue + Employee Self-Service Portal + Europe + Email
Benefits Reimbursement Request + Benefits Portal + North America + Chat

A eso lo llamaremos:

process_unit_id

Así pasamos de 10 procesos generales a muchas unidades operativas analizables.

Esto tiene mucho más sentido ya que, una empresa, no automatiza solo “Payroll” en abstracto. Automatiza partes concretas del proceso según sistema, región, canal, volumen y complejidad.

Esta granularidad permite construir un recomendador más realista y más útil para la toma de decisiones.

In [35]:
# Convertir fechas a formato datetime:
df_hr["ticket_created_date"] = pd.to_datetime(df_hr["ticket_created_date"], errors="coerce")
df_hr["ticket_resolved_date"] = pd.to_datetime(df_hr["ticket_resolved_date"], errors="coerce")

df_hr[["ticket_created_date", "ticket_resolved_date"]].head()


,ticket_created_date,ticket_resolved_date
0,2023-05-17,2023-05-20
1,2024-01-06,2024-01-19
2,2022-11-30,2022-12-05
3,2024-03-21,2024-04-04
4,2024-08-16,2024-08-24


In [36]:
# Crear variables temporales a partir de la fecha de creación del ticket
df_hr["created_year"] = df_hr["ticket_created_date"].dt.year
df_hr["created_month"] = df_hr["ticket_created_date"].dt.month
df_hr["created_dayofweek"] = df_hr["ticket_created_date"].dt.dayofweek
df_hr["created_quarter"] = df_hr["ticket_created_date"].dt.quarter

df_hr[[
    "ticket_created_date",
    "created_year",
    "created_month",
    "created_dayofweek",
    "created_quarter"
]].head()

,ticket_created_date,created_year,created_month,created_dayofweek,created_quarter
0,2023-05-17,2023,5,2,2
1,2024-01-06,2024,1,5,1
2,2022-11-30,2022,11,2,4
3,2024-03-21,2024,3,3,1
4,2024-08-16,2024,8,4,3


In [37]:
# Revisar valores de escalated y sla_breached:
print("Valores de escalated:")
print(df_hr["escalated"].value_counts(dropna=False))

print("\nValores de sla_breached:")
print(df_hr["sla_breached"].value_counts(dropna=False))

Valores de escalated:
escalated
Yes    100421
No      99579
Name: count, dtype: int64

Valores de sla_breached:
sla_breached
Yes    100043
No      99957
Name: count, dtype: int64


In [38]:
# Convertir variables de texto a variables binarias
df_hr["escalated_flag"] = (
    df_hr["escalated"]
    .astype(str)
    .str.lower()
    .str.strip()
    .map({
        "yes": 1,
        "no": 0,
        "true": 1,
        "false": 0,
        "1": 1,
        "0": 0
    })
)

df_hr["sla_breached_flag"] = (
    df_hr["sla_breached"]
    .astype(str)
    .str.lower()
    .str.strip()
    .map({
        "yes": 1,
        "no": 0,
        "true": 1,
        "false": 0,
        "1": 1,
        "0": 0
    })
)

df_hr[["escalated", "escalated_flag", "sla_breached", "sla_breached_flag"]].head()

,escalated,escalated_flag,sla_breached,sla_breached_flag
0,No,0,Yes,1
1,Yes,1,Yes,1
2,Yes,1,Yes,1
3,Yes,1,No,0
4,Yes,1,No,0


In [39]:
# Verificar binarios:
df_hr[["escalated_flag", "sla_breached_flag"]].isna().sum()

escalated_flag       0
sla_breached_flag    0
dtype: int64

In [40]:
# Crear process_unit_id (unidad operativa de proceso):
df_hr["process_unit_id"] = (
    df_hr["hr_process_name"].astype(str) + " | " +
    df_hr["hr_system_name"].astype(str) + " | " +
    df_hr["region"].astype(str) + " | " +
    df_hr["contact_channel"].astype(str) + " | " +
    df_hr["case_priority"].astype(str)
)

df_hr[[
    "hr_process_name",
    "hr_system_name",
    "region",
    "contact_channel",
    "case_priority",
    "process_unit_id"
]].head()

,hr_process_name,hr_system_name,region,contact_channel,case_priority,process_unit_id
0,HRIS Access Suspension,Employee Self-Service Portal,North America,Email,Urgent,HRIS Access Suspension | Employee Self-Service Portal | North America | Email | Urgent
1,HR System Performance Issue,Employee Mobile App,South America,Email,Urgent,HR System Performance Issue | Employee Mobile App | South America | Email | Urgent
2,HR System Performance Issue,Employee Self-Service Portal,Europe,Chat,Medium,HR System Performance Issue | Employee Self-Service Portal | Europe | Chat | Medium
3,Benefits Cancellation Request,Payroll Payment Gateway,Asia,Social Media,Medium,Benefits Cancellation Request | Payroll Payment Gateway | Asia | Social Media | Medium
4,HR System Enhancement Request,Employee Self-Service Portal,Asia,Email,High,HR System Enhancement Request | Employee Self-Service Portal | Asia | Email | High


In [42]:
# Ver cuántas unidades operativas hay:
df_hr["process_unit_id"].nunique()

12000

In [43]:
# Crear resumen por unidad operativa:
# Crear dataset agregado por unidad operativa
process_unit_summary = (
    df_hr
    .groupby([
        "process_unit_id",
        "hr_process_name",
        "hr_system_name",
        "region",
        "contact_channel",
        "case_priority"
    ])
    .agg(
        total_cases=("case_id", "count"),
        avg_resolution_time_hours=("resolution_time_hours", "mean"),
        avg_first_response_time_hours=("first_response_time_hours", "mean"),
        sla_breach_rate=("sla_breached_flag", "mean"),
        escalation_rate=("escalated_flag", "mean"),
        avg_complexity_score=("process_complexity_score", "mean"),
        avg_satisfaction_score=("employee_satisfaction_score", "mean"),
        avg_previous_cases=("previous_cases", "mean"),
        avg_employee_tenure_months=("employee_tenure_months", "mean")
    )
    .reset_index()
)

process_unit_summary.head()

,process_unit_id,hr_process_name,hr_system_name,region,contact_channel,case_priority,total_cases,avg_resolution_time_hours,avg_first_response_time_hours,sla_breach_rate,escalation_rate,avg_complexity_score,avg_satisfaction_score,avg_previous_cases,avg_employee_tenure_months
0,Benefits Cancellation Request | Benefits Administration System | Africa | Chat | High,Benefits Cancellation Request,Benefits Administration System,Africa,Chat,High,17,103.567647,34.694118,0.470588,0.588235,5.823529,3.294118,11.823529,20.529412
1,Benefits Cancellation Request | Benefits Administration System | Africa | Chat | Low,Benefits Cancellation Request,Benefits Administration System,Africa,Chat,Low,19,123.071053,29.458947,0.157895,0.578947,3.473684,2.789474,10.526316,36.842105
2,Benefits Cancellation Request | Benefits Administration System | Africa | Chat | Medium,Benefits Cancellation Request,Benefits Administration System,Africa,Chat,Medium,16,104.613750,38.040000,0.687500,0.437500,5.812500,3.437500,9.437500,41.000000
3,Benefits Cancellation Request | Benefits Administration System | Africa | Chat | Urgent,Benefits Cancellation Request,Benefits Administration System,Africa,Chat,Urgent,19,144.623684,29.998421,0.578947,0.368421,4.842105,2.947368,8.789474,30.526316
4,Benefits Cancellation Request | Benefits Administration System | Africa | Email | High,Benefits Cancellation Request,Benefits Administration System,Africa,Email,High,20,134.300000,38.070500,0.550000,0.550000,4.400000,2.750000,11.250000,24.650000


In [44]:
# Revisar dimensiones del dataset de unidades operativas:
process_unit_summary.shape

(12000, 15)

In [45]:
# Ver unidades con más volumen:
process_unit_summary.sort_values("total_cases", ascending=False).head(20)

,process_unit_id,hr_process_name,hr_system_name,region,contact_channel,case_priority,total_cases,avg_resolution_time_hours,avg_first_response_time_hours,sla_breach_rate,escalation_rate,avg_complexity_score,avg_satisfaction_score,avg_previous_cases,avg_employee_tenure_months
11000,Payroll Support Request | Benefits Portal | North America | Chat | High,Payroll Support Request,Benefits Portal,North America,Chat,High,34,101.190000,39.269118,0.500000,0.588235,5.264706,2.941176,9.323529,30.617647
8293,HR System Performance Issue | People Analytics Dashboard | Africa | Social Media | Low,HR System Performance Issue,People Analytics Dashboard,Africa,Social Media,Low,33,128.416970,39.043030,0.575758,0.484848,6.757576,2.575758,11.484848,27.969697
9940,HRIS Login Issue | Employee Mobile App | South America | Chat | High,HRIS Login Issue,Employee Mobile App,South America,Chat,High,32,124.709063,34.660938,0.531250,0.406250,4.250000,3.281250,10.125000,33.218750
9147,HRIS Access Suspension | HRIS Integration API | Asia | Email | Urgent,HRIS Access Suspension,HRIS Integration API,Asia,Email,Urgent,32,146.628125,42.122813,0.406250,0.718750,5.281250,3.437500,9.562500,30.406250
4743,Employee Data Update | People Analytics Dashboard | Europe | Chat | Urgent,Employee Data Update,People Analytics Dashboard,Europe,Chat,Urgent,31,119.520323,33.368710,0.709677,0.612903,6.064516,2.774194,11.096774,34.354839
6994,HR System Enhancement Request | Payroll System | Asia | Social Media | Medium,HR System Enhancement Request,Payroll System,Asia,Social Media,Medium,31,108.364516,37.021290,0.419355,0.548387,5.645161,2.967742,11.612903,29.806452
9724,HRIS Login Issue | Benefits Portal | Africa | Email | High,HRIS Login Issue,Benefits Portal,Africa,Email,High,31,122.940645,39.267097,0.354839,0.483871,4.935484,3.129032,9.580645,31.903226
6755,HR System Enhancement Request | HRIS Integration API | Asia | Social Media | Urgent,HR System Enhancement Request,HRIS Integration API,Asia,Social Media,Urgent,31,103.363871,34.770000,0.516129,0.548387,6.645161,3.096774,11.903226,28.161290
7193,HR System Enhancement Request | People Analytics Dashboard | South America | Social Media | Low,HR System Enhancement Request,People Analytics Dashboard,South America,Social Media,Low,30,125.621333,35.803667,0.433333,0.600000,5.800000,3.233333,9.833333,28.666667
7229,HR System Performance Issue | Benefits Administration System | Asia | Phone | Low,HR System Performance Issue,Benefits Administration System,Asia,Phone,Low,30,99.621000,38.694000,0.666667,0.466667,4.633333,3.066667,8.900000,32.733333


## Diagnóstico:

Vemos que process_unit_summary tiene:

12.000 filas
15 columnas

Significa que sí conseguimos generar muchas unidades operativas.

Pero hay un problema: las unidades están demasiado fragmentadas.

Uunidades con más volumen tienen solo:

34 casos
33 casos
32 casos
31 casos
30 casos

Eso es poco para calcular métricas estables como:

sla_breach_rate;
escalation_rate;
avg_complexity_score;
avg_satisfaction_score.

Si bien no se considera un error técnico, sí puede ser un problema analítico.

## Decisión Técnica:

Crear una versión mejorada del dataset agregado.

En vez de agrupar por:

proceso + sistema + región + canal + prioridad

se agrupará por:

proceso + sistema + región + canal

Y la prioridad la convertiremos en una métrica interna:

high_priority_rate
urgent_priority_rate

Consieramos esto más profesional ya que la prioridad no debería fragmentar tanto el dataset, sino ayudar a explicar el riesgo y la oportunidad de automatización.

## Ajuste de granularidad de las unidades operativas

La primera versión de las unidades operativas combinaba proceso, sistema, región, canal y prioridad. Aunque esta aproximación generó muchas unidades, también fragmentó demasiado los datos.

Para construir métricas más estables, se ajustará la granularidad eliminando la prioridad como dimensión de agrupación.

La prioridad se mantendrá como variable agregada mediante tasas, por ejemplo:

- porcentaje de casos de prioridad alta;
- porcentaje de casos urgentes.

La unidad operativa final se definirá como:

proceso HR + sistema HR + región + canal de contacto.

In [46]:
# Crear variables de prioridad:
# Crear variables binarias de prioridad
df_hr["high_priority_flag"] = (
    df_hr["case_priority"]
    .astype(str)
    .str.lower()
    .str.strip()
    .eq("high")
    .astype(int)
)

df_hr["urgent_priority_flag"] = (
    df_hr["case_priority"]
    .astype(str)
    .str.lower()
    .str.strip()
    .eq("urgent")
    .astype(int)
)

df_hr[["case_priority", "high_priority_flag", "urgent_priority_flag"]].head()

,case_priority,high_priority_flag,urgent_priority_flag
0,Urgent,0,1
1,Urgent,0,1
2,Medium,0,0
3,Medium,0,0
4,High,1,0


In [47]:
# Crear nueva unidad operativa sin fragmentar por prioridad
df_hr["process_unit_id_v2"] = (
    df_hr["hr_process_name"].astype(str) + " | " +
    df_hr["hr_system_name"].astype(str) + " | " +
    df_hr["region"].astype(str) + " | " +
    df_hr["contact_channel"].astype(str)
)

df_hr[[
    "hr_process_name",
    "hr_system_name",
    "region",
    "contact_channel",
    "process_unit_id_v2"
]].head()

,hr_process_name,hr_system_name,region,contact_channel,process_unit_id_v2
0,HRIS Access Suspension,Employee Self-Service Portal,North America,Email,HRIS Access Suspension | Employee Self-Service Portal | North America | Email
1,HR System Performance Issue,Employee Mobile App,South America,Email,HR System Performance Issue | Employee Mobile App | South America | Email
2,HR System Performance Issue,Employee Self-Service Portal,Europe,Chat,HR System Performance Issue | Employee Self-Service Portal | Europe | Chat
3,Benefits Cancellation Request,Payroll Payment Gateway,Asia,Social Media,Benefits Cancellation Request | Payroll Payment Gateway | Asia | Social Media
4,HR System Enhancement Request,Employee Self-Service Portal,Asia,Email,HR System Enhancement Request | Employee Self-Service Portal | Asia | Email


In [49]:
# Crear dataset agregado por unidad operativa final
process_unit_summary_v2 = (
    df_hr
    .groupby([
        "process_unit_id_v2",
        "hr_process_name",
        "hr_system_name",
        "region",
        "contact_channel"
    ])
    .agg(
        total_cases=("case_id", "count"),
        avg_resolution_time_hours=("resolution_time_hours", "mean"),
        avg_first_response_time_hours=("first_response_time_hours", "mean"),
        sla_breach_rate=("sla_breached_flag", "mean"),
        escalation_rate=("escalated_flag", "mean"),
        avg_complexity_score=("process_complexity_score", "mean"),
        avg_satisfaction_score=("employee_satisfaction_score", "mean"),
        avg_previous_cases=("previous_cases", "mean"),
        avg_employee_tenure_months=("employee_tenure_months", "mean"),
        high_priority_rate=("high_priority_flag", "mean"),
        urgent_priority_rate=("urgent_priority_flag", "mean")
    )
    .reset_index()
)

process_unit_summary_v2.head()

,process_unit_id_v2,hr_process_name,hr_system_name,region,contact_channel,total_cases,avg_resolution_time_hours,avg_first_response_time_hours,sla_breach_rate,escalation_rate,avg_complexity_score,avg_satisfaction_score,avg_previous_cases,avg_employee_tenure_months,high_priority_rate,urgent_priority_rate
0,Benefits Cancellation Request | Benefits Administration System | Africa | Chat,Benefits Cancellation Request,Benefits Administration System,Africa,Chat,71,120.009437,32.790563,0.464789,0.492958,4.929577,3.098592,10.126761,32.183099,0.239437,0.267606
1,Benefits Cancellation Request | Benefits Administration System | Africa | Email,Benefits Cancellation Request,Benefits Administration System,Africa,Email,80,132.294500,36.935875,0.550000,0.475000,5.187500,2.787500,9.750000,24.712500,0.250000,0.237500
2,Benefits Cancellation Request | Benefits Administration System | Africa | Phone,Benefits Cancellation Request,Benefits Administration System,Africa,Phone,58,115.596724,35.816897,0.517241,0.586207,5.793103,3.172414,10.379310,36.862069,0.224138,0.362069
3,Benefits Cancellation Request | Benefits Administration System | Africa | Social Media,Benefits Cancellation Request,Benefits Administration System,Africa,Social Media,57,116.853684,35.772281,0.473684,0.543860,5.596491,2.912281,10.912281,25.684211,0.157895,0.385965
4,Benefits Cancellation Request | Benefits Administration System | Africa | Web Form,Benefits Cancellation Request,Benefits Administration System,Africa,Web Form,62,125.060323,36.214677,0.532258,0.467742,4.983871,2.822581,10.467742,29.983871,0.290323,0.322581


In [50]:
# Revisar dimensiones del dataset de unidades operativas:
process_unit_summary_v2.shape

(3000, 16)

In [51]:
# Ver unidades con más volumen de casos:
process_unit_summary_v2.sort_values("total_cases", ascending=False).head(20)

,process_unit_id_v2,hr_process_name,hr_system_name,region,contact_channel,total_cases,avg_resolution_time_hours,avg_first_response_time_hours,sla_breach_rate,escalation_rate,avg_complexity_score,avg_satisfaction_score,avg_previous_cases,avg_employee_tenure_months,high_priority_rate,urgent_priority_rate
1688,HR System Enhancement Request | HRIS Integration API | Asia | Social Media,HR System Enhancement Request,HRIS Integration API,Asia,Social Media,94,111.472234,39.045851,0.531915,0.510638,5.595745,3.117021,10.074468,31.361702,0.191489,0.329787
2470,HRIS Login Issue | Employee Mobile App | Australia | Chat,HRIS Login Issue,Employee Mobile App,Australia,Chat,94,123.534681,34.880426,0.510638,0.468085,5.468085,2.936170,10.244681,29.808511,0.255319,0.202128
908,Employee Data Update | Benefits Administration System | Asia | Social Media,Employee Data Update,Benefits Administration System,Asia,Social Media,92,127.400761,31.256087,0.532609,0.586957,5.402174,2.891304,9.760870,30.402174,0.250000,0.271739
2300,HRIS Access Suspension | HRIS Integration API | North America | Chat,HRIS Access Suspension,HRIS Integration API,North America,Chat,92,122.725978,41.967065,0.500000,0.521739,5.576087,3.108696,9.554348,31.108696,0.271739,0.239130
176,Benefits Cancellation Request | HR Document Management System | South America | Email,Benefits Cancellation Request,HR Document Management System,South America,Email,91,102.279560,33.505275,0.472527,0.516484,5.571429,3.109890,8.714286,29.857143,0.219780,0.274725
528,Benefits Reimbursement Request | Payroll Payment Gateway | Europe | Social Media,Benefits Reimbursement Request,Payroll Payment Gateway,Europe,Social Media,91,114.162527,37.748462,0.494505,0.516484,5.494505,3.120879,9.274725,30.428571,0.252747,0.219780
754,Compliance & Access Security Review | HR Document Management System | Africa | Web Form,Compliance & Access Security Review,HR Document Management System,Africa,Web Form,91,124.135275,30.670989,0.461538,0.472527,5.373626,2.736264,9.923077,32.406593,0.263736,0.241758
601,Compliance & Access Security Review | Benefits Administration System | Africa | Email,Compliance & Access Security Review,Benefits Administration System,Africa,Email,91,113.724286,35.884615,0.472527,0.505495,5.714286,3.065934,11.109890,29.164835,0.274725,0.263736
838,Compliance & Access Security Review | Payroll Payment Gateway | South America | Social Media,Compliance & Access Security Review,Payroll Payment Gateway,South America,Social Media,91,121.234066,34.479231,0.461538,0.516484,5.758242,2.989011,9.659341,30.373626,0.241758,0.263736
349,Benefits Reimbursement Request | Benefits Portal | Europe | Web Form,Benefits Reimbursement Request,Benefits Portal,Europe,Web Form,90,123.497111,36.118556,0.488889,0.400000,4.900000,3.177778,9.688889,31.277778,0.322222,0.188889


In [52]:
# Comparar granularidad anterior vs nueva:
comparison_granularity = pd.DataFrame({
    "dataset": ["process_unit_summary", "process_unit_summary_v2"],
    "filas": [process_unit_summary.shape[0], process_unit_summary_v2.shape[0]],
    "casos_promedio_por_unidad": [
        process_unit_summary["total_cases"].mean(),
        process_unit_summary_v2["total_cases"].mean()
    ],
    "max_casos_por_unidad": [
        process_unit_summary["total_cases"].max(),
        process_unit_summary_v2["total_cases"].max()
    ],
    "min_casos_por_unidad": [
        process_unit_summary["total_cases"].min(),
        process_unit_summary_v2["total_cases"].min()
    ]
})

comparison_granularity

,dataset,filas,casos_promedio_por_unidad,max_casos_por_unidad,min_casos_por_unidad
0,process_unit_summary,12000,16.666667,34,3
1,process_unit_summary_v2,3000,66.666667,94,37


## Validación de la nueva tabla

| Dataset | Filas | Casos promedio por unidad | Máximo de casos por unidad | Mínimo de casos por unidad |
|---------|-------|--------------------------|---------------------------|---------------------------|
| process_unit_summary | 12.000 | 16,67 | 34 | 3 |
| process_unit_summary_v2 | 3.000 | 66,67 | 94 | 37 |

**`process_unit_summary_v2` es la tabla correcta para continuar** — cada unidad operativa tiene muchos más casos, por lo tanto las métricas son más estables.

## Creación del Automation Opportunity Score

El Automation Opportunity Score mide qué unidades operativas tienen mayor potencial para ser automatizadas u optimizadas con IA.

El score combina variables de volumen, tiempo, fricción operativa, riesgo de SLA, escalación, complejidad y prioridad.

La lógica de negocio es la siguiente:

- más volumen implica mayor oportunidad de impacto;
- mayor tiempo de resolución implica mayor carga operativa;
- mayor tiempo de primera respuesta indica posible cuello de botella;
- mayor tasa de incumplimiento de SLA indica riesgo operativo;
- mayor tasa de escalación indica fricción o complejidad;
- mayor complejidad indica necesidad de optimización;
- mayor proporción de casos urgentes o de prioridad alta indica mayor criticidad.

El objetivo no es automatizar todo, sino priorizar dónde una intervención tecnológica podría generar más valor.

In [53]:
# Crear copia de trabajo para calcular el score de automatización
df_score = process_unit_summary_v2.copy()

df_score.head()

,process_unit_id_v2,hr_process_name,hr_system_name,region,contact_channel,total_cases,avg_resolution_time_hours,avg_first_response_time_hours,sla_breach_rate,escalation_rate,avg_complexity_score,avg_satisfaction_score,avg_previous_cases,avg_employee_tenure_months,high_priority_rate,urgent_priority_rate
0,Benefits Cancellation Request | Benefits Administration System | Africa | Chat,Benefits Cancellation Request,Benefits Administration System,Africa,Chat,71,120.009437,32.790563,0.464789,0.492958,4.929577,3.098592,10.126761,32.183099,0.239437,0.267606
1,Benefits Cancellation Request | Benefits Administration System | Africa | Email,Benefits Cancellation Request,Benefits Administration System,Africa,Email,80,132.294500,36.935875,0.550000,0.475000,5.187500,2.787500,9.750000,24.712500,0.250000,0.237500
2,Benefits Cancellation Request | Benefits Administration System | Africa | Phone,Benefits Cancellation Request,Benefits Administration System,Africa,Phone,58,115.596724,35.816897,0.517241,0.586207,5.793103,3.172414,10.379310,36.862069,0.224138,0.362069
3,Benefits Cancellation Request | Benefits Administration System | Africa | Social Media,Benefits Cancellation Request,Benefits Administration System,Africa,Social Media,57,116.853684,35.772281,0.473684,0.543860,5.596491,2.912281,10.912281,25.684211,0.157895,0.385965
4,Benefits Cancellation Request | Benefits Administration System | Africa | Web Form,Benefits Cancellation Request,Benefits Administration System,Africa,Web Form,62,125.060323,36.214677,0.532258,0.467742,4.983871,2.822581,10.467742,29.983871,0.290323,0.322581


In [55]:
# Normalizamos las variables numéricas para calcular el score de automatización - Esta función transforma cada variable a escala 0-1:
def min_max_scale(series):
    """
    Normaliza una serie numérica entre 0 y 1.
    Si todos los valores son iguales, devuelve 0 para evitar división por cero.
    """
    min_value = series.min()
    max_value = series.max()
    
    if max_value == min_value:
        return series * 0
    
    return (series - min_value) / (max_value - min_value)

In [56]:
# Normalizar variables del score:
# Normalizar variables operativas relevantes
df_score["volume_score"] = min_max_scale(df_score["total_cases"])
df_score["resolution_time_score"] = min_max_scale(df_score["avg_resolution_time_hours"])
df_score["first_response_score"] = min_max_scale(df_score["avg_first_response_time_hours"])
df_score["sla_risk_score"] = min_max_scale(df_score["sla_breach_rate"])
df_score["escalation_score"] = min_max_scale(df_score["escalation_rate"])
df_score["complexity_score"] = min_max_scale(df_score["avg_complexity_score"])
df_score["previous_cases_score"] = min_max_scale(df_score["avg_previous_cases"])
df_score["high_priority_score"] = min_max_scale(df_score["high_priority_rate"])
df_score["urgent_priority_score"] = min_max_scale(df_score["urgent_priority_rate"])

df_score[[
    "volume_score",
    "resolution_time_score",
    "first_response_score",
    "sla_risk_score",
    "escalation_score",
    "complexity_score",
    "previous_cases_score",
    "high_priority_score",
    "urgent_priority_score"
]].head()

,volume_score,resolution_time_score,first_response_score,sla_risk_score,escalation_score,complexity_score,previous_cases_score,high_priority_score,urgent_priority_score
0,0.596491,0.461670,0.355604,0.388291,0.499440,0.216508,0.553901,0.497463,0.461319
1,0.754386,0.666144,0.576636,0.583255,0.460557,0.325110,0.477269,0.524752,0.386505
2,0.368421,0.388225,0.516971,0.508303,0.701346,0.580107,0.605269,0.457940,0.696064
3,0.350877,0.409146,0.514592,0.408644,0.609654,0.497321,0.713674,0.286808,0.755447
4,0.438596,0.545738,0.538181,0.542661,0.444841,0.239369,0.623256,0.628922,0.597934


## Calcular Automation Opportunity Score

Usaremos pesos de negocio — no todos los factores pesan igual.

| Factor | Peso | Motivo |
|--------|------|--------|
| Volumen | 20% | Automatizar procesos frecuentes genera más impacto |
| Tiempo de resolución | 15% | Procesos lentos consumen más capacidad |
| Incumplimiento SLA | 15% | Riesgo operativo directo |
| Escalación | 10% | Indica fricción y necesidad de intervención |
| Complejidad | 10% | Señala procesos difíciles o cargados manualmente |
| Tiempo de primera respuesta | 10% | Mide cuello de botella inicial |
| Casos previos | 5% | Indica recurrencia o historial operativo |
| Prioridad alta | 7.5% | Indica criticidad |
| Prioridad urgente | 7.5% | Indica criticidad máxima |

In [57]:
# Calcular Automation Opportunity Score con pesos de negocio
df_score["automation_score"] = (
    df_score["volume_score"] * 0.20 +
    df_score["resolution_time_score"] * 0.15 +
    df_score["sla_risk_score"] * 0.15 +
    df_score["escalation_score"] * 0.10 +
    df_score["complexity_score"] * 0.10 +
    df_score["first_response_score"] * 0.10 +
    df_score["previous_cases_score"] * 0.05 +
    df_score["high_priority_score"] * 0.075 +
    df_score["urgent_priority_score"] * 0.075
) * 100

df_score["automation_score"] = df_score["automation_score"].round(2)

df_score[[
    "process_unit_id_v2",
    "total_cases",
    "automation_score"
]].sort_values("automation_score", ascending=False).head(20)

,process_unit_id_v2,total_cases,automation_score
2028,HR System Performance Issue | Payroll Payment Gateway | Europe | Social Media,89,66.45
248,Benefits Cancellation Request | Payroll System | Asia | Social Media,84,65.56
317,Benefits Reimbursement Request | Benefits Administration System | Europe | Phone,74,65.23
366,Benefits Reimbursement Request | Employee Mobile App | Asia | Email,76,64.50
1923,HR System Performance Issue | HR Case Management Platform | Africa | Social Media,75,64.38
2915,Payroll Support Request | Payroll Payment Gateway | Asia | Chat,82,64.38
1332,HR Platform Bug Report | HR Case Management Platform | Australia | Phone,71,63.87
407,Benefits Reimbursement Request | Employee Self-Service Portal | Europe | Phone,72,63.73
1427,HR Platform Bug Report | Payroll Payment Gateway | Europe | Phone,74,63.42
1620,HR System Enhancement Request | HR Case Management Platform | Africa | Chat,85,63.37


## Crear prioridad de automatización

Clasificaremos las unidades en tres niveles:

- Alta: score igual o superior a 70.

- Media: score entre 50 y 69.99.

- Baja: score menor de 50.

In [58]:
# Clasificar prioridad de automatización según el score
df_score["automation_priority"] = pd.cut(
    df_score["automation_score"],
    bins=[-1, 50, 70, 100],
    labels=["Baja", "Media", "Alta"]
)

df_score["automation_priority"].value_counts()

automation_priority
Baja     1612
Media    1388
Alta        0
Name: count, dtype: int64

In [59]:
# Crear recomendación de solución tecnológica basada en el proceso y sistema:
def recommend_automation_solution(row):
    """
    Recomienda un tipo de solución de automatización según las características operativas.
    """
    if row["automation_score"] >= 70:
        if row["avg_complexity_score"] >= 6.5 and row["sla_breach_rate"] >= 0.5:
            return "IA asistida con revisión humana"
        elif row["contact_channel"] in ["Email", "Web Form"] and row["avg_complexity_score"] < 6:
            return "Workflow automation"
        elif row["contact_channel"] in ["Chat", "Social Media"] and row["avg_complexity_score"] < 6:
            return "Chatbot o asistente virtual"
        else:
            return "Automatización prioritaria"
    
    elif row["automation_score"] >= 50:
        if row["avg_complexity_score"] >= 6:
            return "Rediseño del proceso antes de automatizar"
        elif row["sla_breach_rate"] >= 0.5:
            return "Automatización parcial con alertas SLA"
        else:
            return "Automatización parcial"
    
    else:
        if row["avg_complexity_score"] >= 7:
            return "Mantener revisión humana"
        else:
            return "Baja prioridad de automatización"


df_score["recommended_solution"] = df_score.apply(recommend_automation_solution, axis=1)

df_score["recommended_solution"].value_counts()

recommended_solution
Baja prioridad de automatización             1610
Automatización parcial con alertas SLA        821
Automatización parcial                        407
Rediseño del proceso antes de automatizar     162
Name: count, dtype: int64

## Estimar horas ahorrables

Supuesto inicial: si una unidad operativa se automatiza parcialmente, se podría ahorrar un porcentaje del tiempo de resolución según su prioridad de automatización.

| Prioridad | Porcentaje de ahorro estimado |
|-----------|-------------------------------|
| Alta | 35% |
| Media | 20% |
| Baja | 5% |

In [62]:
# Definir porcentaje estimado de ahorro según prioridad de automatización
savings_rate_mapping = {
    "Alta": 0.35,
    "Media": 0.20,
    "Baja": 0.05
}

# Crear tasa estimada de ahorro
df_score["estimated_savings_rate"] = (
    df_score["automation_priority"]
    .astype(str)
    .map(savings_rate_mapping)
)

# Calcular horas estimadas ahorrables
df_score["estimated_hours_saved"] = (
    df_score["total_cases"] *
    df_score["avg_resolution_time_hours"] *
    df_score["estimated_savings_rate"]
).round(2)

df_score[[
    "process_unit_id_v2",
    "automation_priority",
    "automation_score",
    "estimated_savings_rate",
    "estimated_hours_saved"
]].sort_values("estimated_hours_saved", ascending=False).head(20)

,process_unit_id_v2,automation_priority,automation_score,estimated_savings_rate,estimated_hours_saved
2921,Payroll Support Request | Payroll Payment Gateway | Australia | Email,Media,59.69,0.2,2378.40
294,Benefits Cancellation Request | People Analytics Dashboard | North America | Web Form,Media,57.98,0.2,2376.64
908,Employee Data Update | Benefits Administration System | Asia | Social Media,Media,60.01,0.2,2344.17
2470,HRIS Login Issue | Employee Mobile App | Australia | Chat,Media,57.93,0.2,2322.45
2248,HRIS Access Suspension | HR Case Management Platform | South America | Social Media,Media,61.45,0.2,2300.22
2559,HRIS Login Issue | HR Document Management System | Asia | Web Form,Media,57.19,0.2,2271.32
754,Compliance & Access Security Review | HR Document Management System | Africa | Web Form,Media,53.37,0.2,2259.26
2300,HRIS Access Suspension | HRIS Integration API | North America | Chat,Media,62.36,0.2,2258.16
2865,Payroll Support Request | HR Document Management System | Europe | Chat,Media,55.85,0.2,2228.02
349,Benefits Reimbursement Request | Benefits Portal | Europe | Web Form,Media,53.05,0.2,2222.95


In [63]:
# Crear ranking final de oportunidades de automatización
automation_ranking = (
    df_score
    .sort_values(
        by=["automation_score", "estimated_hours_saved"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

automation_ranking["ranking"] = automation_ranking.index + 1

automation_ranking[[
    "ranking",
    "hr_process_name",
    "hr_system_name",
    "region",
    "contact_channel",
    "total_cases",
    "automation_score",
    "automation_priority",
    "recommended_solution",
    "estimated_hours_saved"
]].head(20)


,ranking,hr_process_name,hr_system_name,region,contact_channel,total_cases,automation_score,automation_priority,recommended_solution,estimated_hours_saved
0,1,HR System Performance Issue,Payroll Payment Gateway,Europe,Social Media,89,66.45,Media,Rediseño del proceso antes de automatizar,2185.40
1,2,Benefits Cancellation Request,Payroll System,Asia,Social Media,84,65.56,Media,Rediseño del proceso antes de automatizar,2037.86
2,3,Benefits Reimbursement Request,Benefits Administration System,Europe,Phone,74,65.23,Media,Automatización parcial con alertas SLA,1984.51
3,4,Benefits Reimbursement Request,Employee Mobile App,Asia,Email,76,64.50,Media,Rediseño del proceso antes de automatizar,2136.08
4,5,Payroll Support Request,Payroll Payment Gateway,Asia,Chat,82,64.38,Media,Automatización parcial con alertas SLA,2000.57
5,6,HR System Performance Issue,HR Case Management Platform,Africa,Social Media,75,64.38,Media,Rediseño del proceso antes de automatizar,1992.89
6,7,HR Platform Bug Report,HR Case Management Platform,Australia,Phone,71,63.87,Media,Rediseño del proceso antes de automatizar,1830.46
7,8,Benefits Reimbursement Request,Employee Self-Service Portal,Europe,Phone,72,63.73,Media,Rediseño del proceso antes de automatizar,1796.98
8,9,HR Platform Bug Report,Payroll Payment Gateway,Europe,Phone,74,63.42,Media,Automatización parcial con alertas SLA,1912.60
9,10,HR System Enhancement Request,HR Case Management Platform,Africa,Chat,85,63.37,Media,Automatización parcial con alertas SLA,2153.18


## Proceso técnico hasta aqui:
1. Crear df_score
2. Normalizar variables
3. Calcular automation_score
4. Crear automation_priority
5. Crear recommended_solution
6. Crear estimated_savings_rate
7. Crear estimated_hours_saved
8. Crear automation_ranking

## Ajuste de la prioridad de automatización mediante percentiles

El primer cálculo del Automation Opportunity Score generó puntuaciones válidas, pero los umbrales fijos no eran adecuados porque ninguna unidad operativa superaba el umbral de 70 puntos.

Para evitar una clasificación artificial, se utilizarán percentiles.

La lógica será:

- prioridad alta: unidades en el 25% superior del score;
- prioridad media: unidades entre el percentil 25 y 75;
- prioridad baja: unidades en el 25% inferior.

Este enfoque permite priorizar las mejores oportunidades relativas dentro del conjunto de procesos analizados.

In [64]:
# Revisar distribución del score:
df_score["automation_score"].describe()

count    3000.00000
mean       49.44051
std         5.02094
min        31.86000
25%        46.00750
50%        49.59500
75%        52.92000
max        66.45000
Name: automation_score, dtype: float64

In [65]:
# Calcular percentiles:
low_threshold = df_score["automation_score"].quantile(0.25)
high_threshold = df_score["automation_score"].quantile(0.75)

print("Umbral prioridad baja:", round(low_threshold, 2))
print("Umbral prioridad alta:", round(high_threshold, 2))

Umbral prioridad baja: 46.01
Umbral prioridad alta: 52.92


In [66]:
# Reclasificar prioridad de automatización:
def classify_automation_priority(score):
    """
    Clasifica la prioridad de automatización usando percentiles.
    """
    if score >= high_threshold:
        return "Alta"
    elif score >= low_threshold:
        return "Media"
    else:
        return "Baja"


df_score["automation_priority"] = df_score["automation_score"].apply(classify_automation_priority)

df_score["automation_priority"].value_counts()

automation_priority
Media    1497
Alta      753
Baja      750
Name: count, dtype: int64

In [67]:
# Recalcular tasa de ahorro según nueva clasificación - Ahora que sí tendremos prioridad Alta, recalculamos ahorro:
savings_rate_mapping = {
    "Alta": 0.35,
    "Media": 0.20,
    "Baja": 0.05
}

df_score["estimated_savings_rate"] = (
    df_score["automation_priority"]
    .map(savings_rate_mapping)
)

df_score["estimated_hours_saved"] = (
    df_score["total_cases"] *
    df_score["avg_resolution_time_hours"] *
    df_score["estimated_savings_rate"]
).round(2)

df_score[[
    "process_unit_id_v2",
    "automation_priority",
    "automation_score",
    "estimated_savings_rate",
    "estimated_hours_saved"
]].sort_values("automation_score", ascending=False).head(20)

,process_unit_id_v2,automation_priority,automation_score,estimated_savings_rate,estimated_hours_saved
2028,HR System Performance Issue | Payroll Payment Gateway | Europe | Social Media,Alta,66.45,0.35,3824.45
248,Benefits Cancellation Request | Payroll System | Asia | Social Media,Alta,65.56,0.35,3566.26
317,Benefits Reimbursement Request | Benefits Administration System | Europe | Phone,Alta,65.23,0.35,3472.89
366,Benefits Reimbursement Request | Employee Mobile App | Asia | Email,Alta,64.50,0.35,3738.15
1923,HR System Performance Issue | HR Case Management Platform | Africa | Social Media,Alta,64.38,0.35,3487.56
2915,Payroll Support Request | Payroll Payment Gateway | Asia | Chat,Alta,64.38,0.35,3501.00
1332,HR Platform Bug Report | HR Case Management Platform | Australia | Phone,Alta,63.87,0.35,3203.31
407,Benefits Reimbursement Request | Employee Self-Service Portal | Europe | Phone,Alta,63.73,0.35,3144.72
1427,HR Platform Bug Report | Payroll Payment Gateway | Europe | Phone,Alta,63.42,0.35,3347.05
1620,HR System Enhancement Request | HR Case Management Platform | Africa | Chat,Alta,63.37,0.35,3768.06


In [68]:
# Recalcular recomendación de solución tecnológica con nueva prioridad:
def recommend_automation_solution(row):
    """
    Recomienda un tipo de solución de automatización según prioridad, complejidad,
    canal y riesgo operativo.
    """
    if row["automation_priority"] == "Alta":
        if row["avg_complexity_score"] >= 6.5 and row["sla_breach_rate"] >= 0.5:
            return "IA asistida con revisión humana"
        elif row["contact_channel"] in ["Email", "Web Form"] and row["avg_complexity_score"] < 6:
            return "Workflow automation"
        elif row["contact_channel"] in ["Chat", "Social Media"] and row["avg_complexity_score"] < 6:
            return "Chatbot o asistente virtual"
        elif row["sla_breach_rate"] >= 0.5:
            return "Automatización parcial con alertas SLA"
        else:
            return "Automatización prioritaria"

    elif row["automation_priority"] == "Media":
        if row["avg_complexity_score"] >= 6:
            return "Rediseño del proceso antes de automatizar"
        elif row["sla_breach_rate"] >= 0.5:
            return "Automatización parcial con alertas SLA"
        else:
            return "Automatización parcial"

    else:
        if row["avg_complexity_score"] >= 7:
            return "Mantener revisión humana"
        else:
            return "Baja prioridad de automatización"


df_score["recommended_solution"] = df_score.apply(recommend_automation_solution, axis=1)

df_score["recommended_solution"].value_counts()

recommended_solution
Automatización parcial con alertas SLA       900
Baja prioridad de automatización             750
Automatización parcial                       685
Chatbot o asistente virtual                  269
Workflow automation                          243
Rediseño del proceso antes de automatizar     91
Automatización prioritaria                    62
Name: count, dtype: int64

In [69]:
# Rehacer ranking final de oportunidades de automatización con nueva prioridad y recomendaciones:
automation_ranking = (
    df_score
    .sort_values(
        by=["automation_score", "estimated_hours_saved"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

automation_ranking["ranking"] = automation_ranking.index + 1

automation_ranking[[
    "ranking",
    "hr_process_name",
    "hr_system_name",
    "region",
    "contact_channel",
    "total_cases",
    "automation_score",
    "automation_priority",
    "recommended_solution",
    "estimated_hours_saved"
]].head(20)

,ranking,hr_process_name,hr_system_name,region,contact_channel,total_cases,automation_score,automation_priority,recommended_solution,estimated_hours_saved
0,1,HR System Performance Issue,Payroll Payment Gateway,Europe,Social Media,89,66.45,Alta,Automatización parcial con alertas SLA,3824.45
1,2,Benefits Cancellation Request,Payroll System,Asia,Social Media,84,65.56,Alta,Automatización parcial con alertas SLA,3566.26
2,3,Benefits Reimbursement Request,Benefits Administration System,Europe,Phone,74,65.23,Alta,Automatización parcial con alertas SLA,3472.89
3,4,Benefits Reimbursement Request,Employee Mobile App,Asia,Email,76,64.50,Alta,Automatización parcial con alertas SLA,3738.15
4,5,Payroll Support Request,Payroll Payment Gateway,Asia,Chat,82,64.38,Alta,Chatbot o asistente virtual,3501.00
5,6,HR System Performance Issue,HR Case Management Platform,Africa,Social Media,75,64.38,Alta,Automatización parcial con alertas SLA,3487.56
6,7,HR Platform Bug Report,HR Case Management Platform,Australia,Phone,71,63.87,Alta,Automatización parcial con alertas SLA,3203.31
7,8,Benefits Reimbursement Request,Employee Self-Service Portal,Europe,Phone,72,63.73,Alta,Automatización parcial con alertas SLA,3144.72
8,9,HR Platform Bug Report,Payroll Payment Gateway,Europe,Phone,74,63.42,Alta,Automatización parcial con alertas SLA,3347.05
9,10,HR System Enhancement Request,HR Case Management Platform,Africa,Chat,85,63.37,Alta,Chatbot o asistente virtual,3768.06


In [70]:
# Resumen actualizado: 
executive_summary = pd.DataFrame({
    "métrica": [
        "Unidades operativas analizadas",
        "Score promedio de automatización",
        "Score máximo",
        "Score mínimo",
        "Unidades de prioridad alta",
        "Unidades de prioridad media",
        "Unidades de prioridad baja",
        "Horas estimadas ahorrables totales"
    ],
    "valor": [
        len(df_score),
        round(df_score["automation_score"].mean(), 2),
        round(df_score["automation_score"].max(), 2),
        round(df_score["automation_score"].min(), 2),
        int((df_score["automation_priority"] == "Alta").sum()),
        int((df_score["automation_priority"] == "Media").sum()),
        int((df_score["automation_priority"] == "Baja").sum()),
        round(df_score["estimated_hours_saved"].sum(), 2)
    ]
})

executive_summary


,métrica,valor
0,Unidades operativas analizadas,3000.00
1,Score promedio de automatización,49.44
2,Score máximo,66.45
3,Score mínimo,31.86
4,Unidades de prioridad alta,753.00
5,Unidades de prioridad media,1497.00
6,Unidades de prioridad baja,750.00
7,Horas estimadas ahorrables totales,5055719.93


## Diagnóstico

El ranking está bien estructurado:

automation_score funciona.
automation_priority clasifica en Alta / Media / Baja.
recommended_solution genera recomendaciones.
automation_ranking se creó correctamente.

Pero Horas estimadas ahorrables totales no funciona: 5.055.719,93

Es demasiado alto para presentarlo como “horas ahorrables”. 

El problema viene de esta fórmula:

total_cases * avg_resolution_time_hours * estimated_savings_rate

Estamos usando resolution_time_hours, que representa el tiempo total hasta resolver el caso. Pero ese tiempo incluye espera, colas, pausas, dependencia de otros equipos, retrasos y tiempo calendario. No equivale a tiempo manual real de trabajo.

Necesitamos crear una estimación realista:

no ahorramos un porcentaje del tiempo total de resolución, sino del tiempo operativo/manual estimado.

Por este motivo crearemos una variable nueva:

estimated_manual_handling_time_hours



## Ajuste de la estimación de ahorro operativo

La primera estimación de horas ahorrables utilizaba el tiempo total de resolución del caso. Sin embargo, el tiempo de resolución incluye esperas, colas, dependencias externas y tiempo calendario.

Para construir una estimación más realista, se calculará primero un tiempo operativo/manual estimado.

La lógica será:

- los procesos de prioridad alta tendrán mayor proporción de trabajo manual potencialmente optimizable;
- los procesos de prioridad media tendrán una proporción intermedia;
- los procesos de baja prioridad tendrán menor potencial de ahorro.

Esta corrección permite estimar ahorro operativo desde una perspectiva de negocio.

In [71]:
# Definir proporción estimada de trabajo manual sobre el tiempo total de resolución
manual_work_rate_mapping = {
    "Alta": 0.18,
    "Media": 0.12,
    "Baja": 0.08
}

df_score["estimated_manual_work_rate"] = (
    df_score["automation_priority"]
    .map(manual_work_rate_mapping)
)

df_score[[
    "automation_priority",
    "estimated_manual_work_rate"
]].drop_duplicates()

,automation_priority,estimated_manual_work_rate
0,Baja,0.08
1,Alta,0.18
2,Media,0.12


In [72]:
# Calcular tiempo manual estimado por caso:
df_score["estimated_manual_handling_time_hours"] = (
    df_score["avg_resolution_time_hours"] *
    df_score["estimated_manual_work_rate"]
).round(2)

df_score[[
    "avg_resolution_time_hours",
    "automation_priority",
    "estimated_manual_work_rate",
    "estimated_manual_handling_time_hours"
]].head()

,avg_resolution_time_hours,automation_priority,estimated_manual_work_rate,estimated_manual_handling_time_hours
0,120.009437,Baja,0.08,9.60
1,132.294500,Alta,0.18,23.81
2,115.596724,Media,0.12,13.87
3,116.853684,Media,0.12,14.02
4,125.060323,Media,0.12,15.01


In [73]:
# Recalcular horas ahorrables - mantenemos el ahorro estimado por prioridad, pero ahora aplicado sobre el tiempo manual estimado:
# Mantener tasa de ahorro estimado según prioridad
savings_rate_mapping = {
    "Alta": 0.35,
    "Media": 0.20,
    "Baja": 0.05
}

df_score["estimated_savings_rate"] = (
    df_score["automation_priority"]
    .map(savings_rate_mapping)
)

# Calcular horas operativas estimadas ahorrables
df_score["estimated_operational_hours_saved"] = (
    df_score["total_cases"] *
    df_score["estimated_manual_handling_time_hours"] *
    df_score["estimated_savings_rate"]
).round(2)

df_score[[
    "process_unit_id_v2",
    "automation_priority",
    "automation_score",
    "total_cases",
    "avg_resolution_time_hours",
    "estimated_manual_handling_time_hours",
    "estimated_savings_rate",
    "estimated_operational_hours_saved"
]].sort_values("estimated_operational_hours_saved", ascending=False).head(20)


,process_unit_id_v2,automation_priority,automation_score,total_cases,avg_resolution_time_hours,estimated_manual_handling_time_hours,estimated_savings_rate,estimated_operational_hours_saved
2921,Payroll Support Request | Payroll Payment Gateway | Australia | Email,Alta,59.69,86,138.279070,24.89,0.35,749.19
294,Benefits Cancellation Request | People Analytics Dashboard | North America | Web Form,Alta,57.98,89,133.518989,24.03,0.35,748.53
908,Employee Data Update | Benefits Administration System | Asia | Social Media,Alta,60.01,92,127.400761,22.93,0.35,738.35
2470,HRIS Login Issue | Employee Mobile App | Australia | Chat,Alta,57.93,94,123.534681,22.24,0.35,731.70
2248,HRIS Access Suspension | HR Case Management Platform | South America | Social Media,Alta,61.45,87,132.196437,23.80,0.35,724.71
2559,HRIS Login Issue | HR Document Management System | Asia | Web Form,Alta,57.19,85,133.607176,24.05,0.35,715.49
754,Compliance & Access Security Review | HR Document Management System | Africa | Web Form,Alta,53.37,91,124.135275,22.34,0.35,711.53
2300,HRIS Access Suspension | HRIS Integration API | North America | Chat,Alta,62.36,92,122.725978,22.09,0.35,711.30
2865,Payroll Support Request | HR Document Management System | Europe | Chat,Alta,55.85,88,126.592159,22.79,0.35,701.93
349,Benefits Reimbursement Request | Benefits Portal | Europe | Web Form,Alta,53.05,90,123.497111,22.23,0.35,700.24


In [74]:
# Crear ranking final corregido de oportunidades de automatización
automation_ranking = (
    df_score
    .sort_values(
        by=["automation_score", "estimated_operational_hours_saved"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

automation_ranking["ranking"] = automation_ranking.index + 1

automation_ranking[[
    "ranking",
    "hr_process_name",
    "hr_system_name",
    "region",
    "contact_channel",
    "total_cases",
    "automation_score",
    "automation_priority",
    "recommended_solution",
    "estimated_operational_hours_saved"
]].head(20)

,ranking,hr_process_name,hr_system_name,region,contact_channel,total_cases,automation_score,automation_priority,recommended_solution,estimated_operational_hours_saved
0,1,HR System Performance Issue,Payroll Payment Gateway,Europe,Social Media,89,66.45,Alta,Automatización parcial con alertas SLA,688.42
1,2,Benefits Cancellation Request,Payroll System,Asia,Social Media,84,65.56,Alta,Automatización parcial con alertas SLA,641.80
2,3,Benefits Reimbursement Request,Benefits Administration System,Europe,Phone,74,65.23,Alta,Automatización parcial con alertas SLA,625.23
3,4,Benefits Reimbursement Request,Employee Mobile App,Asia,Email,76,64.50,Alta,Automatización parcial con alertas SLA,672.98
4,5,Payroll Support Request,Payroll Payment Gateway,Asia,Chat,82,64.38,Alta,Chatbot o asistente virtual,630.25
5,6,HR System Performance Issue,HR Case Management Platform,Africa,Social Media,75,64.38,Alta,Automatización parcial con alertas SLA,627.64
6,7,HR Platform Bug Report,HR Case Management Platform,Australia,Phone,71,63.87,Alta,Automatización parcial con alertas SLA,576.52
7,8,Benefits Reimbursement Request,Employee Self-Service Portal,Europe,Phone,72,63.73,Alta,Automatización parcial con alertas SLA,565.99
8,9,HR Platform Bug Report,Payroll Payment Gateway,Europe,Phone,74,63.42,Alta,Automatización parcial con alertas SLA,602.43
9,10,HR System Enhancement Request,HR Case Management Platform,Africa,Chat,85,63.37,Alta,Chatbot o asistente virtual,678.30


In [75]:
# Resumen corregido:
executive_summary = pd.DataFrame({
    "métrica": [
        "Unidades operativas analizadas",
        "Score promedio de automatización",
        "Score máximo",
        "Score mínimo",
        "Unidades de prioridad alta",
        "Unidades de prioridad media",
        "Unidades de prioridad baja",
        "Horas operativas estimadas ahorrables totales"
    ],
    "valor": [
        len(df_score),
        round(df_score["automation_score"].mean(), 2),
        round(df_score["automation_score"].max(), 2),
        round(df_score["automation_score"].min(), 2),
        int((df_score["automation_priority"] == "Alta").sum()),
        int((df_score["automation_priority"] == "Media").sum()),
        int((df_score["automation_priority"] == "Baja").sum()),
        round(df_score["estimated_operational_hours_saved"].sum(), 2)
    ]
})

executive_summary

,métrica,valor
0,Unidades operativas analizadas,3000.00
1,Score promedio de automatización,49.44
2,Score máximo,66.45
3,Score mínimo,31.86
4,Unidades de prioridad alta,753.00
5,Unidades de prioridad media,1497.00
6,Unidades de prioridad baja,750.00
7,Horas operativas estimadas ahorrables totales,739371.90


## Justificación técnica:

Antes decíamos:

“Si un caso tarda 120 horas en resolverse, automatizarlo ahorra una parte de esas 120 horas.”

No es realista.

Ahora decimos:

“Si un caso tarda 120 horas en resolverse, solo una parte corresponde a trabajo manual estimado. La automatización puede reducir una parte de ese trabajo operativo.”



## Ajuste de la estimación de ahorro operativo

La primera estimación de horas ahorrables utilizaba el tiempo total de resolución del caso. Sin embargo, el tiempo de resolución incluye esperas, colas, dependencias externas y tiempo calendario.

Por este motivo, se ajustará la estimación calculando primero un tiempo operativo/manual estimado.

La lógica será:

- no todo el tiempo de resolución puede ser reducido mediante automatización;
- solo una parte del tiempo corresponde a trabajo operativo o manual;
- la automatización reduce una proporción de ese trabajo manual estimado;
- el ahorro se calcula sobre tiempo operativo estimado, no sobre el tiempo total de resolución.

Con este ajuste logramos, que el impacto estimado, sea más realista desde una perspectiva de negocio.

In [76]:
# Definir proporción estimada de trabajo manual sobre el tiempo total de resolución
manual_work_rate_mapping = {
    "Alta": 0.18,
    "Media": 0.12,
    "Baja": 0.08
}

df_score["estimated_manual_work_rate"] = (
    df_score["automation_priority"]
    .map(manual_work_rate_mapping)
)

df_score[[
    "automation_priority",
    "estimated_manual_work_rate"
]].drop_duplicates().sort_values("automation_priority")

,automation_priority,estimated_manual_work_rate
1,Alta,0.18
0,Baja,0.08
2,Media,0.12


## Qué significa

| Prioridad | Interpretación |
|-----------|----------------|
| Alta | Se asume que el 18% del tiempo total de resolución corresponde a trabajo manual optimizable |
| Media | Se asume un 12% |
| Baja | Se asume un 8% |

In [77]:
# Calcular tiempo manual estimado por caso
df_score["estimated_manual_handling_time_hours"] = (
    df_score["avg_resolution_time_hours"] *
    df_score["estimated_manual_work_rate"]
).round(2)

df_score[[
    "hr_process_name",
    "hr_system_name",
    "region",
    "contact_channel",
    "automation_priority",
    "avg_resolution_time_hours",
    "estimated_manual_work_rate",
    "estimated_manual_handling_time_hours"
]].head(20)

,hr_process_name,hr_system_name,region,contact_channel,automation_priority,avg_resolution_time_hours,estimated_manual_work_rate,estimated_manual_handling_time_hours
0,Benefits Cancellation Request,Benefits Administration System,Africa,Chat,Baja,120.009437,0.08,9.60
1,Benefits Cancellation Request,Benefits Administration System,Africa,Email,Alta,132.294500,0.18,23.81
2,Benefits Cancellation Request,Benefits Administration System,Africa,Phone,Media,115.596724,0.12,13.87
3,Benefits Cancellation Request,Benefits Administration System,Africa,Social Media,Media,116.853684,0.12,14.02
4,Benefits Cancellation Request,Benefits Administration System,Africa,Web Form,Media,125.060323,0.12,15.01
5,Benefits Cancellation Request,Benefits Administration System,Asia,Chat,Media,117.746087,0.12,14.13
6,Benefits Cancellation Request,Benefits Administration System,Asia,Email,Media,130.475139,0.12,15.66
7,Benefits Cancellation Request,Benefits Administration System,Asia,Phone,Baja,114.776129,0.08,9.18
8,Benefits Cancellation Request,Benefits Administration System,Asia,Social Media,Media,121.287612,0.12,14.55
9,Benefits Cancellation Request,Benefits Administration System,Asia,Web Form,Media,124.090323,0.12,14.89


In [78]:
# Definir tasa estimada de ahorro según prioridad de automatización
savings_rate_mapping = {
    "Alta": 0.35,
    "Media": 0.20,
    "Baja": 0.05
}

df_score["estimated_savings_rate"] = (
    df_score["automation_priority"]
    .map(savings_rate_mapping)
)

# Calcular horas operativas estimadas ahorrables
df_score["estimated_operational_hours_saved"] = (
    df_score["total_cases"] *
    df_score["estimated_manual_handling_time_hours"] *
    df_score["estimated_savings_rate"]
).round(2)

df_score[[
    "process_unit_id_v2",
    "automation_priority",
    "automation_score",
    "total_cases",
    "avg_resolution_time_hours",
    "estimated_manual_handling_time_hours",
    "estimated_savings_rate",
    "estimated_operational_hours_saved"
]].sort_values("estimated_operational_hours_saved", ascending=False).head(20)

,process_unit_id_v2,automation_priority,automation_score,total_cases,avg_resolution_time_hours,estimated_manual_handling_time_hours,estimated_savings_rate,estimated_operational_hours_saved
2921,Payroll Support Request | Payroll Payment Gateway | Australia | Email,Alta,59.69,86,138.279070,24.89,0.35,749.19
294,Benefits Cancellation Request | People Analytics Dashboard | North America | Web Form,Alta,57.98,89,133.518989,24.03,0.35,748.53
908,Employee Data Update | Benefits Administration System | Asia | Social Media,Alta,60.01,92,127.400761,22.93,0.35,738.35
2470,HRIS Login Issue | Employee Mobile App | Australia | Chat,Alta,57.93,94,123.534681,22.24,0.35,731.70
2248,HRIS Access Suspension | HR Case Management Platform | South America | Social Media,Alta,61.45,87,132.196437,23.80,0.35,724.71
2559,HRIS Login Issue | HR Document Management System | Asia | Web Form,Alta,57.19,85,133.607176,24.05,0.35,715.49
754,Compliance & Access Security Review | HR Document Management System | Africa | Web Form,Alta,53.37,91,124.135275,22.34,0.35,711.53
2300,HRIS Access Suspension | HRIS Integration API | North America | Chat,Alta,62.36,92,122.725978,22.09,0.35,711.30
2865,Payroll Support Request | HR Document Management System | Europe | Chat,Alta,55.85,88,126.592159,22.79,0.35,701.93
349,Benefits Reimbursement Request | Benefits Portal | Europe | Web Form,Alta,53.05,90,123.497111,22.23,0.35,700.24


In [79]:
# Crear ranking final corregido de oportunidades de automatización
automation_ranking = (
    df_score
    .sort_values(
        by=["automation_score", "estimated_operational_hours_saved"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

automation_ranking["ranking"] = automation_ranking.index + 1

automation_ranking[[
    "ranking",
    "hr_process_name",
    "hr_system_name",
    "region",
    "contact_channel",
    "total_cases",
    "automation_score",
    "automation_priority",
    "recommended_solution",
    "estimated_operational_hours_saved"
]].head(20)

,ranking,hr_process_name,hr_system_name,region,contact_channel,total_cases,automation_score,automation_priority,recommended_solution,estimated_operational_hours_saved
0,1,HR System Performance Issue,Payroll Payment Gateway,Europe,Social Media,89,66.45,Alta,Automatización parcial con alertas SLA,688.42
1,2,Benefits Cancellation Request,Payroll System,Asia,Social Media,84,65.56,Alta,Automatización parcial con alertas SLA,641.80
2,3,Benefits Reimbursement Request,Benefits Administration System,Europe,Phone,74,65.23,Alta,Automatización parcial con alertas SLA,625.23
3,4,Benefits Reimbursement Request,Employee Mobile App,Asia,Email,76,64.50,Alta,Automatización parcial con alertas SLA,672.98
4,5,Payroll Support Request,Payroll Payment Gateway,Asia,Chat,82,64.38,Alta,Chatbot o asistente virtual,630.25
5,6,HR System Performance Issue,HR Case Management Platform,Africa,Social Media,75,64.38,Alta,Automatización parcial con alertas SLA,627.64
6,7,HR Platform Bug Report,HR Case Management Platform,Australia,Phone,71,63.87,Alta,Automatización parcial con alertas SLA,576.52
7,8,Benefits Reimbursement Request,Employee Self-Service Portal,Europe,Phone,72,63.73,Alta,Automatización parcial con alertas SLA,565.99
8,9,HR Platform Bug Report,Payroll Payment Gateway,Europe,Phone,74,63.42,Alta,Automatización parcial con alertas SLA,602.43
9,10,HR System Enhancement Request,HR Case Management Platform,Africa,Chat,85,63.37,Alta,Chatbot o asistente virtual,678.30


In [80]:
# Crear resumen ejecutivo corregido
executive_summary = pd.DataFrame({
    "métrica": [
        "Unidades operativas analizadas",
        "Score promedio de automatización",
        "Score máximo",
        "Score mínimo",
        "Unidades de prioridad alta",
        "Unidades de prioridad media",
        "Unidades de prioridad baja",
        "Horas operativas estimadas ahorrables totales"
    ],
    "valor": [
        len(df_score),
        round(df_score["automation_score"].mean(), 2),
        round(df_score["automation_score"].max(), 2),
        round(df_score["automation_score"].min(), 2),
        int((df_score["automation_priority"] == "Alta").sum()),
        int((df_score["automation_priority"] == "Media").sum()),
        int((df_score["automation_priority"] == "Baja").sum()),
        round(df_score["estimated_operational_hours_saved"].sum(), 2)
    ]
})

executive_summary

,métrica,valor
0,Unidades operativas analizadas,3000.00
1,Score promedio de automatización,49.44
2,Score máximo,66.45
3,Score mínimo,31.86
4,Unidades de prioridad alta,753.00
5,Unidades de prioridad media,1497.00
6,Unidades de prioridad baja,750.00
7,Horas operativas estimadas ahorrables totales,739371.90


In [81]:
# Comparar estimación anterior con estimación corregida
savings_comparison = pd.DataFrame({
    "métrica": [
        "Horas ahorrables estimadas originalmente",
        "Horas operativas ahorrables corregidas",
        "Reducción de la estimación"
    ],
    "valor": [
        round(df_score["estimated_hours_saved"].sum(), 2),
        round(df_score["estimated_operational_hours_saved"].sum(), 2),
        round(
            df_score["estimated_hours_saved"].sum() -
            df_score["estimated_operational_hours_saved"].sum(),
            2
        )
    ]
})

savings_comparison

,métrica,valor
0,Horas ahorrables estimadas originalmente,5055719.93
1,Horas operativas ahorrables corregidas,739371.90
2,Reducción de la estimación,4316348.03


### Interpretación del ajuste

La estimación inicial de ahorro utilizaba el tiempo total de resolución, lo que podía sobreestimar el impacto real de la automatización.

Para corregirlo, se estimó primero qué parte del tiempo total corresponde a trabajo operativo/manual. Posteriormente, el ahorro potencial se calculó únicamente sobre esa fracción manual estimada.

Este enfoque es más conservador. Reconoce que no todo el tiempo de resolución puede ser reducido mediante automatización. Parte del tiempo corresponde a esperas, dependencias externas, colas de trabajo o tiempos administrativos que no necesariamente desaparecen al automatizar.

## Resumen de resultados

| Métrica | Valor |
|---------|-------|
| Unidades operativas analizadas | 3.000 |
| Score promedio de automatización | 49,44 |
| Score máximo | 66,45 |
| Score mínimo | 31,86 |
| Unidades de prioridad alta | 753 |
| Unidades de prioridad media | 1.497 |
| Unidades de prioridad baja | 750 |
| Horas operativas estimadas ahorrables totales | 739.371,90 |

La distribución tiene sentido:

| Prioridad | Proporción |
|-----------|------------|
| Alta | ~25% |
| Media | ~50% |
| Baja | ~25% |

Eso confirma que el ajuste por percentiles funcionó correctamente.

## Validación del ajuste



| Métrica | Valor |
|---------|-------|
| Horas ahorrables estimadas originalmente | 5.055.719,93 |
| Horas operativas ahorrables corregidas | 739.371,90 |
| Reducción de la estimación | 4.316.348,03 |







> "La automatización reduce una parte estimada del trabajo operativo/manual asociado a cada proceso."

In [82]:
# Guardar ranking final de oportunidades de automatización
automation_ranking.to_csv(
    FINAL_DATA_PATH / "hr_process_automation_ranking.csv",
    index=False
)

print("Archivo guardado correctamente en:")
print(FINAL_DATA_PATH / "hr_process_automation_ranking.csv")

Archivo guardado correctamente en:
/Users/gabrielbohorquez/Desktop/Ironhack/PROYECTO_FINAL/HR-Process-Automation-Scanner/data/final/hr_process_automation_ranking.csv


In [83]:
# Guardar resumen ejecutivo
executive_summary.to_csv(
    FINAL_DATA_PATH / "executive_summary.csv",
    index=False
)

print("Archivo guardado correctamente en:")
print(FINAL_DATA_PATH / "executive_summary.csv")

Archivo guardado correctamente en:
/Users/gabrielbohorquez/Desktop/Ironhack/PROYECTO_FINAL/HR-Process-Automation-Scanner/data/final/executive_summary.csv


In [84]:
# Guardar comparación de estimaciones de ahorro
savings_comparison.to_csv(
    FINAL_DATA_PATH / "savings_comparison.csv",
    index=False
)

print("Archivo guardado correctamente en:")
print(FINAL_DATA_PATH / "savings_comparison.csv")

Archivo guardado correctamente en:
/Users/gabrielbohorquez/Desktop/Ironhack/PROYECTO_FINAL/HR-Process-Automation-Scanner/data/final/savings_comparison.csv


In [85]:
# Verificar guardado de archivos:
for file_path in FINAL_DATA_PATH.glob("*.csv"):
    print(file_path.name)

hr_process_automation_ranking.csv
savings_comparison.csv
executive_summary.csv


## Conclusiones del notebook de comprensión de datos

En este notebook se completó la primera fase del proyecto: comprensión, adaptación y preparación inicial de los datos.

A partir de un dataset público de tickets de soporte, se construyó una capa analítica adaptada al contexto de HR Operations. Cada ticket fue interpretado como una solicitud interna de Recursos Humanos y las categorías originales fueron transformadas en procesos y sistemas propios del ecosistema HR.

Durante esta fase se realizaron los siguientes pasos:

- carga e inspección del dataset principal;
- eliminación de columnas no relevantes para el objetivo del proyecto;
- adaptación de categorías originales a procesos de HR Operations;
- adaptación de sistemas originales a plataformas HR;
- creación de unidades operativas de proceso;
- ajuste de granularidad para mejorar la estabilidad de las métricas;
- cálculo del Automation Opportunity Score;
- clasificación de unidades operativas en prioridad alta, media y baja;
- generación de recomendaciones de automatización;
- estimación corregida de horas operativas ahorrables;
- exportación de los datasets finales para las siguientes fases del proyecto.

La tabla final `hr_process_automation_ranking.csv` será utilizada como base para el análisis exploratorio, visualizaciones, modelo de Machine Learning y posterior aplicación en Streamlit.

El resultado de esta fase es un dataset analítico preparado para responder la pregunta principal del proyecto:

¿Qué procesos de HR Operations tienen mayor oportunidad de ser automatizados u optimizados con IA, considerando volumen, tiempos, SLA, escalaciones, complejidad y potencial de ahorro operativo?